# NSG-IDS: Neuro-Symbolic Generative Framework for Intrusion Detection
**Full implementation of the NSG-IDS paper.**
## Data Requirements

> **IMPORTANT:** This notebook requires real IDS datasets. If no datasets are found,
> it will **raise an error** rather than silently fall back to demo data.
> To allow demo mode (NOT suitable for paper results), set `CFG['ALLOW_DEMO_MODE'] = True`.

Datasets expected in `./dataset/` sub-directories:
- `./dataset/CIC-IDS-2017/` — any CSV with CICFlowMeter columns + `Label`
- `./dataset/UNSW-NB15/` — any CSV with UNSW-NB15 columns + `attack_cat`
- `./dataset/NF-ToN-IoT/` — any CSV with NetFlow columns + `Label`

If no files are found, realistic demo data is generated automatically.

In [1]:
import os, warnings, json, time, subprocess, sys



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.mixture import BayesianGaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.manifold import TSNE
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import spectral_norm
from tqdm import tqdm
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}')



# ── reproducibility ──
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Reproducibility vs. speed: the cuDNN autotuner (benchmark) picks different
# kernels per run, which breaks determinism. Set REPRODUCIBLE=False to prioritise
# raw speed; keep True for the paper's reproducible final runs.
# MATCH_BACKENDS=True pins ONE training recipe (same batch size, full fp32, no
# AMP, no TF32) so CUDA and MPS run the identical algorithm and give matching
# results. Set False to let each backend use its fastest settings (bigger batch
# + AMP/TF32 on CUDA), trading cross-backend equivalence for speed.
MATCH_BACKENDS = False

REPRODUCIBLE = True
torch.backends.cudnn.deterministic = REPRODUCIBLE
torch.backends.cudnn.benchmark     = (not REPRODUCIBLE) and torch.cuda.is_available()
if torch.cuda.is_available():
    # TF32 (Ampere+) accelerates matmuls at reduced precision that MPS lacks,
    # so keep it off when matching backends to preserve fp32 on both.
    _tf32 = not MATCH_BACKENDS
    torch.backends.cuda.matmul.allow_tf32 = _tf32
    torch.backends.cudnn.allow_tf32      = _tf32

# ── output dirs ──
for d in ['./outputs', './models', './figures']:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

Device: cuda
Torch: 2.8.0+cu128 | CUDA: True | MPS: False
Directories ready.


In [2]:
# ══════════════════════════════════════════════════════════════
#  CONFIGURATION  (edit here to tune)
# ══════════════════════════════════════════════════════════════
CFG = dict(
    # paths
    DATASET_DIR   = '../dataset',
    # CIF-32 preprocessing
    K_VGM         = 10,   # Gaussian components per continuous feature
    N_BINS        = 32,   # equipopulation bins for discrete-int features
    # architecture
    Z_DIM         = 128,  # latent noise dim
    EMBED_DIM     = 64,   # class-source embedding dim
    HIDDEN_DIM    = 512,
    N_RES_BLOCKS  = 4,
    ATTN_HEADS    = 8,
    ATTN_FEAT_DIM = 32,   # d_e per CIF feature after reshape for MHA
    DROPOUT       = 0.1,
    # training
    BATCH_SIZE    = 512,
    N_EPOCHS      = 100,
    CRITIC_STEPS  = 5,
    LAMBDA_GP     = 1.0,
    GP_EVERY      = 1,    # GP every critic step (standard WGAN-GP)
    LR            = 5e-5,
    BETAS         = (0.5, 0.9),
    EMA_DECAY     = 0.999,  # EMA of G weights for sampling (0 disables)
    FM_WEIGHT     = 0.0,    # feature/moment-matching aux loss on G (ablation showed it hurts; 0 = off)
    TAU_START     = 1.0,
    TAU_END       = 0.2,
    # evaluation
    N_SYNTH_PER_CLASS = 20_000,
    N_RUNS            = 3,
    COVERAGE_MIN      = 500,
    RF_TREES          = 500,
    # ── performance / GPU
    # NUM_WORKERS: DataLoader prefetch workers (0 = main process; 4 is good for most machines)
    NUM_WORKERS  = 0,  # 0 = main process (required for notebook; workers can't unpickle __main__ classes)
    # PIN_MEMORY: pinned CPU memory → faster CPU→GPU DMA transfers (CUDA only)
    PIN_MEMORY   = __import__('torch').cuda.is_available(),
    # USE_AMP: automatic mixed precision fp16/bf16 (CUDA only; ~2-3x speedup)
    USE_AMP      = __import__('torch').cuda.is_available(),
    # BATCH_SIZE_GPU: override BATCH_SIZE when CUDA is present (larger batches = better GPU utilisation)
    BATCH_SIZE_GPU = 1024,  # bf16 halves activations; drop back to 512 if CUDA OOM on 6 GB
    # demo / fast-test
    DEMO_ROWS_PER_SRC = 8_000,   # rows per source when no real data found
    FAST_EPOCHS       = 0,      # set > 0 to override N_EPOCHS for quick test; 0 = full N_EPOCHS
    # ── data fraction: change DATA_PCT to 0.10 | 0.20 | 0.50 | 1.00
    DATA_PCT          = 0.50,    # fraction of training data to use
)
# ── auto-scale batch size for GPU
import torch as _torch
MATCH_BACKENDS = globals().get('MATCH_BACKENDS', True)  # safety if setup cell not re-run
if MATCH_BACKENDS:
    CFG['USE_AMP'] = False  # full fp32 on every backend -> CUDA and MPS match
    print(f'MATCH_BACKENDS=True -> BATCH_SIZE={CFG["BATCH_SIZE"]} on all backends, AMP=False, fp32')
elif _torch.cuda.is_available():
    CFG['BATCH_SIZE'] = CFG['BATCH_SIZE_GPU']
    print(f'CUDA detected → BATCH_SIZE={CFG["BATCH_SIZE"]}, AMP={CFG["USE_AMP"]}, '
          f'NUM_WORKERS={CFG["NUM_WORKERS"]}')
else:
    print(f'No CUDA → BATCH_SIZE={CFG["BATCH_SIZE"]}, device will be MPS or CPU')
# ══════════════════════════════════════════════════════════════
#  SMOKE TEST  —  fast "is the model actually learning?" pass.
#  Run this BEFORE committing to the multi-day full run. When True:
#    • short schedule + tiny data slice  -> finishes in minutes
#    • frequent KPI probe                -> watch KS/CD/JSD drop live
#    • ablation (cell 26) + baselines (cell 28) skip themselves
#  The MODEL RECIPE (arch, K_VGM, EMA, GP, LR ...) is left UNTOUCHED,
#  so the fidelity TREND is representative of the full run — if KS
#  is not trending down here, the full run will not fix it.
#  >>> Set SMOKE_TEST = False for the real paper run. <<<
# ══════════════════════════════════════════════════════════════
SMOKE_TEST = False
if SMOKE_TEST:
    CFG.update(
        FAST_EPOCHS       = 20,      # short run; the eval probe shows the trend
        DATA_PCT          = 0.10,    # tiny slice — enough to see learning start
        N_SYNTH_PER_CLASS = 2_000,   # cheaper fidelity / TSTR eval
        N_RUNS            = 1,        # one eval pass instead of 3
        RF_TREES          = 50,      # lighter TSTR random forest
    )
    SMOKE_EVAL_EVERY = 5             # KPI probe cadence during the smoke run
    print('*** SMOKE_TEST ON ***  epochs=20, data=5%, ablation/baselines SKIPPED.'
          '  Set SMOKE_TEST=False for the full run.')
else:
    SMOKE_EVAL_EVERY = 20
print('Config loaded.')

CUDA detected → BATCH_SIZE=1024, AMP=True, NUM_WORKERS=0
Config loaded.


## 1 · Data Loading & CIF-32 Harmonisation

In [3]:
# ── CIF-32 column definitions ──────────────────────────────────
# Each entry: (cif_name, type)  where type ∈ {'cat','cont','int','bin'}
CIF32 = [
    ('protocol',         'cat'),   # 0
    ('src_port_cat',     'cat'),   # 1
    ('dst_port_cat',     'cat'),   # 2
    ('flow_duration',    'cont'),  # 3
    ('iat_mean',         'cont'),  # 4
    ('iat_std',          'cont'),  # 5
    ('fwd_pkts',         'int'),   # 6
    ('fwd_bytes',        'int'),   # 7
    ('fwd_pkt_len_mean', 'cont'),  # 8
    ('fwd_pkt_len_std',  'cont'),  # 9
    ('bwd_pkts',         'int'),   # 10
    ('bwd_bytes',        'int'),   # 11
    ('bwd_pkt_len_mean', 'cont'),  # 12
    ('bwd_pkt_len_std',  'cont'),  # 13
    ('bytes_per_sec',    'cont'),  # 14
    ('pkts_per_sec',     'cont'),  # 15
    ('fwd_bwd_ratio',   'cont'),   # 16
    ('down_up_ratio',   'cont'),   # 17
    ('syn_flag',         'bin'),   # 18
    ('fin_flag',         'bin'),   # 19
    ('rst_flag',         'bin'),   # 20
    ('psh_flag',         'bin'),   # 21
    ('ack_flag',         'bin'),   # 22
    ('urg_flag',         'bin'),   # 23
    ('subflow_fwd_pkts', 'int'),   # 24
    ('subflow_bwd_pkts', 'int'),   # 25
    ('init_win_fwd',     'int'),   # 26
    ('init_win_bwd',     'int'),   # 27
    ('ttl_fwd',          'cont'),  # 28
    ('ttl_bwd',          'cont'),  # 29
    ('fwd_bulk_rate',    'cont'),  # 30
    ('bwd_bulk_rate',    'cont'),  # 31
]
CIF_NAMES  = [c for c,_ in CIF32]
CIF_TYPES  = {c:t for c,t in CIF32}
CONT_COLS  = [c for c,t in CIF32 if t=='cont']
INT_COLS   = [c for c,t in CIF32 if t=='int']
CAT_COLS   = [c for c,t in CIF32 if t=='cat']
BIN_COLS   = [c for c,t in CIF32 if t=='bin']
print(f'CIF-32: {len(CONT_COLS)} cont, {len(INT_COLS)} int, {len(CAT_COLS)} cat, {len(BIN_COLS)} bin')

# Protocol map (derived from port / proto number)
def proto_cat(p):
    p = str(p).strip().upper()
    if p in ('6','TCP'):   return 0
    if p in ('17','UDP'):  return 1
    return 2  # ICMP / other

def port_cat(p):
    try: p = int(float(p))
    except: return 2
    if p < 1024:   return 0  # well-known
    if p < 49152:  return 1  # registered
    return 2                 # ephemeral

CIF-32: 15 cont, 8 int, 3 cat, 6 bin


In [4]:
# ── per-source CIF-32 mapping functions ────────────────────────

def map_cic2017(df):
    """Map CIC-IDS-2017 CICFlowMeter columns → CIF-32.
    Handles both full names ('Total Fwd Packets') and
    abbreviated names ('Tot Fwd Pkts', 'Flow Byts/s', 'Init Fwd Win Byts', etc.)."""
    df.columns = df.columns.str.strip()
    label_col = next((c for c in df.columns if c.strip().lower() == 'label'), df.columns[-1])
    df = df.replace([float('inf'), float('-inf')], float('nan')).dropna(subset=[label_col])

    ALIASES = {
        'Protocol':                    ['Protocol'],
        'Source Port':                 ['Source Port', 'Src Port', 'SrcPort'],
        'Destination Port':            ['Destination Port', 'Dst Port', 'DestPort'],
        'Flow Duration':               ['Flow Duration'],
        'Flow IAT Mean':               ['Flow IAT Mean'],
        'Flow IAT Std':                ['Flow IAT Std'],
        'Total Fwd Packets':           ['Total Fwd Packets', 'Tot Fwd Pkts'],
        'Total Length of Fwd Packets': ['Total Length of Fwd Packets', 'TotLen Fwd Pkts'],
        'Fwd Packet Length Mean':      ['Fwd Packet Length Mean', 'Fwd Pkt Len Mean'],
        'Fwd Packet Length Std':       ['Fwd Packet Length Std', 'Fwd Pkt Len Std'],
        'Total Backward Packets':      ['Total Backward Packets', 'Tot Bwd Pkts'],
        'Total Length of Bwd Packets': ['Total Length of Bwd Packets', 'TotLen Bwd Pkts'],
        'Bwd Packet Length Mean':      ['Bwd Packet Length Mean', 'Bwd Pkt Len Mean'],
        'Bwd Packet Length Std':       ['Bwd Packet Length Std', 'Bwd Pkt Len Std'],
        'Flow Bytes/s':                ['Flow Bytes/s', 'Flow Byts/s'],
        'Flow Packets/s':              ['Flow Packets/s', 'Flow Pkts/s'],
        'Down/Up Ratio':               ['Down/Up Ratio'],
        'SYN Flag Count':              ['SYN Flag Count', 'SYN Flag Cnt'],
        'FIN Flag Count':              ['FIN Flag Count', 'FIN Flag Cnt'],
        'RST Flag Count':              ['RST Flag Count', 'RST Flag Cnt'],
        'PSH Flag Count':              ['PSH Flag Count', 'PSH Flag Cnt'],
        'ACK Flag Count':              ['ACK Flag Count', 'ACK Flag Cnt'],
        'URG Flag Count':              ['URG Flag Count', 'URG Flag Cnt'],
        'Subflow Fwd Packets':         ['Subflow Fwd Packets', 'Subflow Fwd Pkts'],
        'Subflow Bwd Packets':         ['Subflow Bwd Packets', 'Subflow Bwd Pkts'],
        'Init_Win_bytes_forward':      ['Init_Win_bytes_forward', 'Init Fwd Win Byts'],
        'Init_Win_bytes_backward':     ['Init_Win_bytes_backward', 'Init Bwd Win Byts'],
        'Fwd Avg Bulk Rate':           ['Fwd Avg Bulk Rate', 'Fwd Blk Rate Avg'],
        'Bwd Avg Bulk Rate':           ['Bwd Avg Bulk Rate', 'Bwd Blk Rate Avg'],
    }
    col_map = {c.strip().lower().replace(' ', '').replace('_', ''): c for c in df.columns}

    def gc(canonical, default=0.0):
        for alias in ALIASES.get(canonical, [canonical]):
            key = alias.lower().replace(' ', '').replace('_', '')
            if key in col_map:
                return df[col_map[key]].fillna(default).astype(float)
        return pd.Series(default, index=df.index)

    out = pd.DataFrame()
    out['protocol']         = gc('Protocol', 6).apply(proto_cat)
    out['src_port_cat']     = gc('Source Port', 0).apply(port_cat)
    out['dst_port_cat']     = gc('Destination Port', 0).apply(port_cat)
    dur                     = gc('Flow Duration', 1).clip(lower=1e-6)
    out['flow_duration']    = dur
    out['iat_mean']         = gc('Flow IAT Mean', 0)
    out['iat_std']          = gc('Flow IAT Std', 0)
    out['fwd_pkts']         = gc('Total Fwd Packets', 0)
    out['fwd_bytes']        = gc('Total Length of Fwd Packets', 0)
    out['fwd_pkt_len_mean'] = gc('Fwd Packet Length Mean', 0)
    out['fwd_pkt_len_std']  = gc('Fwd Packet Length Std', 0)
    out['bwd_pkts']         = gc('Total Backward Packets', 0)
    out['bwd_bytes']        = gc('Total Length of Bwd Packets', 0)
    out['bwd_pkt_len_mean'] = gc('Bwd Packet Length Mean', 0)
    out['bwd_pkt_len_std']  = gc('Bwd Packet Length Std', 0)
    out['bytes_per_sec']    = gc('Flow Bytes/s', 0).replace([float('inf'), float('-inf')], 0)
    out['pkts_per_sec']     = gc('Flow Packets/s', 0).replace([float('inf'), float('-inf')], 0)
    # Ratio computed on raw packet counts before any clipping to avoid zero-bias
    out['fwd_bwd_ratio']    = out['fwd_pkts'].clip(lower=0) / (out['bwd_pkts'].clip(lower=0) + 1)
    out['down_up_ratio']    = gc('Down/Up Ratio', 1)
    out['syn_flag']         = (gc('SYN Flag Count', 0) > 0).astype(int)
    out['fin_flag']         = (gc('FIN Flag Count', 0) > 0).astype(int)
    out['rst_flag']         = (gc('RST Flag Count', 0) > 0).astype(int)
    out['psh_flag']         = (gc('PSH Flag Count', 0) > 0).astype(int)
    out['ack_flag']         = (gc('ACK Flag Count', 0) > 0).astype(int)
    out['urg_flag']         = (gc('URG Flag Count', 0) > 0).astype(int)
    out['subflow_fwd_pkts'] = gc('Subflow Fwd Packets', 0)
    out['subflow_bwd_pkts'] = gc('Subflow Bwd Packets', 0)
    out['init_win_fwd']     = gc('Init_Win_bytes_forward', 0).clip(lower=0)
    out['init_win_bwd']     = gc('Init_Win_bytes_backward', 0).clip(lower=0)
    out['ttl_fwd']          = 64.0
    out['ttl_bwd']          = 64.0
    out['fwd_bulk_rate']    = gc('Fwd Avg Bulk Rate', 0)
    out['bwd_bulk_rate']    = gc('Bwd Avg Bulk Rate', 0)
    raw_label = df[label_col].fillna('Benign').astype(str).str.strip()
    # Fix truncated/corrupted labels in raw CICFlowMeter files
    raw_label = raw_label.replace({'FTP-Brute': 'FTP-BruteForce', 'Do': 'DoS'})
    out['label'] = raw_label
    # Drop rows where label is literally 'Label' (repeated header corruption)
    out = out[out['label'] != 'Label']
    out['source']           = 0
    out['missingness']      = 0
    return out.reset_index(drop=True)


def map_unsw(df):
    """Map UNSW-NB15 columns → CIF-32."""
    df.columns = df.columns.str.strip().str.lower()
    df = df.replace([float('inf'), float('-inf')], float('nan'))
    def gc(name, default=0.0):
        return df[name].fillna(default).astype(float) if name in df.columns else pd.Series(default, index=df.index)
    def gcs(name, default=''):
        return df[name].fillna(default).astype(str) if name in df.columns else pd.Series(default, index=df.index)
    out = pd.DataFrame()
    out['protocol']         = gcs('proto', 'tcp').apply(proto_cat)
    out['src_port_cat']     = gc('sport', 0).apply(port_cat)
    out['dst_port_cat']     = gc('dsport', 0).apply(port_cat)
    out['flow_duration']    = gc('dur', 1).clip(lower=1e-6)
    out['iat_mean']         = gc('sinpkt', 0)
    out['iat_std']          = gc('dinpkt', 0)
    out['fwd_pkts']         = gc('spkts', 0)
    out['fwd_bytes']        = gc('sbytes', 0)
    out['fwd_pkt_len_mean'] = gc('smean', 0)
    out['fwd_pkt_len_std']  = gc('sjit', 0)
    out['bwd_pkts']         = gc('dpkts', 0)
    out['bwd_bytes']        = gc('dbytes', 0)
    out['bwd_pkt_len_mean'] = gc('dmean', 0)
    out['bwd_pkt_len_std']  = gc('djit', 0)
    dur = out['flow_duration'].clip(lower=1e-6)
    out['bytes_per_sec']    = (out['fwd_bytes'] + out['bwd_bytes']) / dur
    out['pkts_per_sec']     = (out['fwd_pkts'] + out['bwd_pkts']) / dur
    out['fwd_bwd_ratio']    = out['fwd_pkts'] / (out['bwd_pkts'] + 1)
    out['down_up_ratio']    = gc('dload', 0) / (gc('sload', 1) + 1e-6)
    state = gcs('state', 'CON')
    out['syn_flag']  = state.str.contains('SYN|INT', case=False).astype(int)
    out['fin_flag']  = state.str.contains('FIN|CLO', case=False).astype(int)
    out['rst_flag']  = state.str.contains('RST',     case=False).astype(int)
    out['psh_flag']  = 0
    out['ack_flag']  = state.str.contains('CON|ECO', case=False).astype(int)
    out['urg_flag']  = 0
    out['subflow_fwd_pkts'] = gc('spkts', 0) // 2
    out['subflow_bwd_pkts'] = gc('dpkts', 0) // 2
    out['init_win_fwd']     = gc('swin', 0)
    out['init_win_bwd']     = gc('dwin', 0)
    out['ttl_fwd']          = gc('sttl', 64)
    out['ttl_bwd']          = gc('dttl', 64)
    out['fwd_bulk_rate']    = gc('sload', 0)
    out['bwd_bulk_rate']    = gc('dload', 0)
    label_col = 'attack_cat' if 'attack_cat' in df.columns else 'label'
    out['label']  = df[label_col].fillna('Normal').astype(str).str.strip().replace({'': 'Normal', 'nan': 'Normal'})
    out['source'] = 1
    out['missingness'] = 0
    return out.reset_index(drop=True)


def map_toniot(df):
    """Map NF-ToN-IoT (original Zeek/Bro 'train_test_network.csv' schema) -> CIF-32.

    IMPORTANT: this file uses the Zeek connection-log schema (src_pkts, dst_bytes,
    conn_state, duration, proto, ...), NOT the NetFlow-v2 names (IN_PKTS, TCP_FLAGS,
    L4_SRC_PORT, ...). The previous version targeted the NetFlow schema, so every
    feature fell back to its constant default -- which is exactly why ToN scored
    KS~0.65, CD=nan and real->real F1~0.001. This version reads the real columns."""
    df.columns = df.columns.str.strip().str.lower()
    df = df.replace([float('inf'), float('-inf')], float('nan'))

    def gc(name, default=0.0):
        return (pd.to_numeric(df[name], errors='coerce').fillna(default).astype(float)
                if name in df.columns else pd.Series(default, index=df.index, dtype=float))
    def gcs(name, default=''):
        return df[name].fillna(default).astype(str) if name in df.columns else pd.Series(default, index=df.index)

    out = pd.DataFrame()
    out['protocol']         = gcs('proto', 'tcp').apply(proto_cat)
    out['src_port_cat']     = gc('src_port', 0).apply(port_cat)
    out['dst_port_cat']     = gc('dst_port', 0).apply(port_cat)
    dur                     = gc('duration', 0).clip(lower=1e-6)
    fwd_pkts                = gc('src_pkts', 0)
    bwd_pkts                = gc('dst_pkts', 0)
    fwd_bytes               = gc('src_bytes', 0)
    bwd_bytes               = gc('dst_bytes', 0)
    out['flow_duration']    = dur
    out['iat_mean']         = dur / (fwd_pkts + bwd_pkts + 1)     # mean inter-arrival proxy
    out['iat_std']          = 0.0                                 # per-packet jitter unavailable
    out['fwd_pkts']         = fwd_pkts
    out['fwd_bytes']        = fwd_bytes
    out['fwd_pkt_len_mean'] = fwd_bytes / (fwd_pkts + 1)
    out['fwd_pkt_len_std']  = 0.0                                 # per-packet stats unavailable
    out['bwd_pkts']         = bwd_pkts
    out['bwd_bytes']        = bwd_bytes
    out['bwd_pkt_len_mean'] = bwd_bytes / (bwd_pkts + 1)
    out['bwd_pkt_len_std']  = 0.0
    out['bytes_per_sec']    = (fwd_bytes + bwd_bytes) / dur
    out['pkts_per_sec']     = (fwd_pkts + bwd_pkts) / dur
    out['fwd_bwd_ratio']    = fwd_pkts / (bwd_pkts + 1)
    out['down_up_ratio']    = bwd_bytes / (fwd_bytes + 1)
    # TCP flags derived from the Zeek conn_state summary (per-flag counts are not
    # in the conn log). Non-TCP flows (proto != tcp) get all flags = 0.
    state  = gcs('conn_state', 'OTH').str.upper()
    is_tcp = out['protocol'].eq(0)
    syn = state.isin(['S0', 'S1', 'S2', 'S3', 'SF', 'SH', 'SHR', 'REJ', 'RSTOS0', 'RSTRH'])
    fin = state.isin(['SF', 'S2', 'S3'])
    rst = state.str.startswith('RST') | state.eq('REJ')
    ack = state.isin(['S1', 'S2', 'S3', 'SF', 'OTH', 'RSTO', 'RSTR', 'RSTRH'])
    out['syn_flag'] = (syn & is_tcp).astype(int)
    out['fin_flag'] = (fin & is_tcp).astype(int)
    out['rst_flag'] = (rst & is_tcp).astype(int)
    out['psh_flag'] = 0
    out['ack_flag'] = (ack & is_tcp).astype(int)
    out['urg_flag'] = 0
    out['subflow_fwd_pkts'] = fwd_pkts // 2
    out['subflow_bwd_pkts'] = bwd_pkts // 2
    out['init_win_fwd']     = gc('src_ip_bytes', 0).clip(lower=0)  # proxy: total fwd IP bytes
    out['init_win_bwd']     = gc('dst_ip_bytes', 0).clip(lower=0)  # proxy: total bwd IP bytes
    out['ttl_fwd']          = 64.0
    out['ttl_bwd']          = 64.0
    out['fwd_bulk_rate']    = fwd_bytes / dur
    out['bwd_bulk_rate']    = bwd_bytes / dur
    # Labels: prefer the multiclass 'type' column over the binary 'label'.
    label_col = next((c for c in df.columns if c.strip().lower() == 'type'),
                     next((c for c in df.columns if c.strip().lower() == 'label'), df.columns[-1]))
    LABEL_MAP = {
        'normal': 'Normal', 'benign': 'Benign', 'backdoor': 'Backdoor', 'ddos': 'DDoS',
        'dos': 'DoS', 'injection': 'Injection', 'mitm': 'MITM', 'password': 'Password',
        'ransomware': 'Ransomware', 'scanning': 'Scanning', 'xss': 'XSS',
    }
    out['label'] = df[label_col].fillna('Normal').astype(str).str.strip().str.lower().map(
        lambda x: LABEL_MAP.get(x, x.title()))
    out['source']      = 2
    out['missingness'] = 1
    return out.reset_index(drop=True)


In [5]:
# ── demo data generator (used when real CSVs not found) ────────
def make_demo_data(n=CFG['DEMO_ROWS_PER_SRC'], seed=42):
    rng = np.random.default_rng(seed)
    src_info = [
        (0, ['BENIGN','DoS Hulk','PortScan','DDoS','FTP-Patator','SSH-Patator',
              'Bot','Infiltration','Heartbleed','XSS','SQL Injection',
              'Brute Force','DoS Slowloris','Web Attack']),
        (1, ['Normal','Fuzzers','Analysis','Backdoors','DoS','Exploits',
              'Generic','Reconnaissance','Shellcode','Worms']),
        (2, ['Benign','DoS','DDoS','Reconnaissance','Backdoor','Injection',
              'MITM','Ransomware','Password','XSS','Scanning']),
    ]
    frames = []
    for src_id, labels in src_info:
        # imbalanced: benign >> attacks
        weights = np.array([10.0] + [1.0]*(len(labels)-1))
        weights /= weights.sum()
        chosen = rng.choice(labels, size=n, p=weights)
        row = {}
        row['protocol']         = rng.integers(0,3,n)
        row['src_port_cat']     = rng.integers(0,3,n)
        row['dst_port_cat']     = rng.integers(0,3,n)
        row['flow_duration']    = rng.exponential(1e5, n)
        row['iat_mean']         = rng.exponential(5e4, n)
        row['iat_std']          = rng.exponential(3e4, n)
        row['fwd_pkts']         = rng.integers(1,500,n)
        row['fwd_bytes']        = rng.integers(0,500000,n)
        row['fwd_pkt_len_mean'] = rng.uniform(0,1500,n)
        row['fwd_pkt_len_std']  = rng.uniform(0,500,n)
        row['bwd_pkts']         = rng.integers(0,400,n)
        row['bwd_bytes']        = rng.integers(0,400000,n)
        row['bwd_pkt_len_mean'] = rng.uniform(0,1500,n)
        row['bwd_pkt_len_std']  = rng.uniform(0,500,n)
        row['bytes_per_sec']    = rng.exponential(1e4, n)
        row['pkts_per_sec']     = rng.exponential(100, n)
        row['fwd_bwd_ratio']    = rng.exponential(2, n)
        row['down_up_ratio']    = rng.exponential(1, n)
        for f in ['syn_flag','fin_flag','rst_flag','psh_flag','ack_flag','urg_flag']:
            row[f] = rng.integers(0,2,n)
        row['subflow_fwd_pkts'] = rng.integers(0,100,n)
        row['subflow_bwd_pkts'] = rng.integers(0,100,n)
        row['init_win_fwd']     = rng.integers(0,65536,n)
        row['init_win_bwd']     = rng.integers(0,65536,n)
        row['ttl_fwd']          = rng.choice([64,128,255], n).astype(float)
        row['ttl_bwd']          = rng.choice([64,128,255], n).astype(float)
        row['fwd_bulk_rate']    = rng.exponential(5000, n)
        row['bwd_bulk_rate']    = rng.exponential(5000, n)
        row['label']            = chosen
        row['source']           = src_id
        row['missingness']      = int(src_id == 2)
        frames.append(pd.DataFrame(row))
    return pd.concat(frames, ignore_index=True)

In [6]:
# ── main loader ────────────────────────────────────────────────
SKIP_CSV_TOKENS = ('feature', 'list_event', 'readme', 'schema')

def read_csv_with_fallback(path):
    for encoding in ('utf-8', 'cp1252', 'latin1'):
        try:
            return pd.read_csv(path, low_memory=False, encoding=encoding)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, low_memory=False)

def load_source(subdir, map_fn, max_rows=None):
    p = Path(CFG['DATASET_DIR']) / subdir
    csvs = list(p.glob('*.csv')) + list(p.glob('*.CSV')) if p.exists() else []
    if not csvs:
        return None
    dfs = []
    for f in csvs:
        lower_name = f.name.lower()
        if any(token in lower_name for token in SKIP_CSV_TOKENS):
            print(f'  Skipped metadata file {f.name}')
            continue
        try:
            chunk = read_csv_with_fallback(f)
            if max_rows: chunk = chunk.sample(min(max_rows, len(chunk)), random_state=SEED)
            dfs.append(map_fn(chunk))
            print(f'  Loaded {f.name}: {len(chunk):,} rows')
        except Exception as e:
            print(f'  Warning: skipped {f.name}: {e}')
    return pd.concat(dfs, ignore_index=True) if dfs else None

print('Loading datasets...')
MAX_ROWS = None  # per CSV; set None for full dataset
cic   = load_source('cicids',  map_cic2017, MAX_ROWS)
unsw  = load_source('unsw',     map_unsw,    MAX_ROWS)
toniot= load_source('ton_iot',    map_toniot,  MAX_ROWS)

if cic is None and unsw is None and toniot is None:
    if not CFG.get('ALLOW_DEMO_MODE', False):
        raise RuntimeError(
            'No real datasets found in ./dataset/.\n'
            'To use synthetic demo data instead (NOT suitable for paper results),\n'
            'set CFG["ALLOW_DEMO_MODE"] = True before re-running this cell.'
        )
    print('\n' + '='*70)
    print('WARNING: Running in DEMO MODE — results are NOT reproducible')
    print('         and must NOT be included in the paper.')
    print('='*70 + '\n')
    CFG['DEMO_MODE'] = True
    fused = make_demo_data()
else:
    parts = [df for df in [cic, unsw, toniot] if df is not None]
    fused = pd.concat(parts, ignore_index=True)
    print(f'Fused corpus: {len(fused):,} rows')

# clip extreme values
for c in CONT_COLS + INT_COLS:
    fused[c] = pd.to_numeric(fused[c], errors='coerce').fillna(0).clip(lower=0)
fused['label']  = fused['label'].astype(str).str.strip()
# merge the three per-source benign labels (CIC 'BENIGN', UNSW 'Normal',
# ToN 'Benign') into one canonical 'Benign' class so benign is not split
# across sources and the attack-class count matches the paper.
fused['label'] = fused['label'].replace({'BENIGN': 'Benign', 'Normal': 'Benign'})
fused['source'] = fused['source'].astype(int)

# train / test split (stratified 80/20 on class×source pairs to prevent leakage)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder as _LE
_strat_key = [f'{l}_{s}' for l, s in zip(fused['label'], fused['source'])]
_strat_enc = _LE().fit_transform(_strat_key)
train_df, test_df = train_test_split(fused, test_size=0.2, stratify=_strat_enc, random_state=SEED)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

# encode class labels — fit ONLY on training set to prevent label leakage
label_enc = LabelEncoder()
train_df['class_id'] = label_enc.fit_transform(train_df['label'])
# map test labels to training encoding (unseen labels → first class)
_known = set(label_enc.classes_)
test_df['label_safe'] = test_df['label'].apply(lambda x: x if x in _known else label_enc.classes_[0])
test_df['class_id']  = label_enc.transform(test_df['label_safe'])
test_df = test_df.drop(columns=['label_safe'])

source_enc = LabelEncoder()
train_df['source_id'] = source_enc.fit_transform(train_df['source'])
test_df['source_id']  = source_enc.transform(test_df['source'])

N_CLASSES = len(label_enc.classes_)
N_SOURCES = len(source_enc.classes_)
print(f'Classes: {N_CLASSES}  |  Sources: {N_SOURCES}')
print(label_enc.classes_)

# ── DATA_PCT subsampling ────────────────────────────────────────────────────
_pct = CFG.get('DATA_PCT', 1.0)
if _pct < 1.0:
    from sklearn.model_selection import train_test_split as _tts
    _, train_df = _tts(train_df, test_size=_pct,
                       stratify=train_df['class_id'], random_state=SEED)
    train_df = train_df.reset_index(drop=True)
    print(f'DATA_PCT={_pct*100:.0f}%  ->  training rows: {len(train_df):,}')
else:
    print(f'DATA_PCT=100%  ->  training rows: {len(train_df):,}')

# per-source test subsets
test_by_src = {s: test_df[test_df['source']==s] for s in test_df['source'].unique()}

Loading datasets...
  Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
  Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
  Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows
  Loaded Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows
  Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows
  Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 rows
  Loaded Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 rows
  Loaded Wednesday-workingHours.pcap_ISCX.csv: 692,703 rows
  Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
  Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
  Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows
  Loaded Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows
  Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows
  Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 rows


## 2 · Preprocessing — VGM, Binning, One-Hot

In [7]:
# ── Variational Gaussian Mixture transformer ───────────────────
class VGMTransformer:
    """VGM normalization per continuous column (CTGAN-style)."""
    def __init__(self, K=5, max_iter=100, scale_factor=4.0):
        self.K = K
        self.scale_factor = scale_factor  # normalisation range: value lands in [-1,1] when within scale_factor std devs
        self.bgm = {}
        self.log_cols = set()  # heavy-tailed cols modelled in log1p space

    def fit(self, df, cols):
        from joblib import Parallel, delayed
        def _fit_one(c):
            v = df[c].values.astype(float).reshape(-1, 1)
            # log1p heavy-tailed non-negative cols (bytes/s, throughput, durations):
            # VGM(K=5) cannot spread modes across several orders of magnitude on raw
            # values, so the bulk collapses into one mode and the tail clips to +/-1.
            med, p99 = np.median(v), np.percentile(v, 99)
            use_log = bool(v.min() >= 0 and p99 > 10.0 * (med + 1e-6))
            if use_log:
                v = np.log1p(v)
            bgm = BayesianGaussianMixture(n_components=self.K, n_init=1,
                                          max_iter=100, random_state=SEED)
            bgm.fit(v)
            return c, bgm, use_log
        # fit all columns in parallel (n_jobs=-1 = all CPU cores)
        results = Parallel(n_jobs=-1, prefer='threads')(delayed(_fit_one)(c) for c in cols)
        self.bgm = {c: bgm for c, bgm, _ in results}
        self.log_cols = {c for c, _, lg in results if lg}
        return self

    def transform(self, df, cols):
        parts = []
        self.col_slices = {}
        offset = 0
        for c in cols:
            bgm = self.bgm[c]
            v   = df[c].values.astype(float).reshape(-1,1)
            if c in self.log_cols:
                v = np.log1p(np.clip(v, 0, None))
            probs = bgm.predict_proba(v)          # (N, K)
            mode  = np.argmax(probs, axis=1)       # (N,)
            means = bgm.means_.flatten()
            stds  = np.sqrt(bgm.covariances_.flatten())
            norm  = (v.flatten() - means[mode]) / (self.scale_factor * stds[mode] + 1e-6)
            norm  = np.clip(norm, -1, 1)
            onehot = np.zeros((len(v), self.K), dtype=np.float32)
            onehot[np.arange(len(v)), mode] = 1.0
            block = np.column_stack([onehot, norm.astype(np.float32)])  # (N, K+1)
            parts.append(block)
            self.col_slices[c] = (offset, offset + self.K + 1)
            offset += self.K + 1
        return np.column_stack(parts) if parts else np.zeros((len(df), 0))

    def inverse(self, arr, cols):
        out = {}
        for c in cols:
            s, e = self.col_slices[c]
            bgm  = self.bgm[c]
            oh   = arr[:, s:s+self.K]
            norm = arr[:, s+self.K]
            mode = np.argmax(oh, axis=1)
            means = bgm.means_.flatten()
            stds  = np.sqrt(bgm.covariances_.flatten())
            val = norm * self.scale_factor * stds[mode] + means[mode]
            out[c] = np.expm1(val) if c in self.log_cols else val
        return out


# ── Equipopulation bin transformer ────────────────────────────
class BinTransformer:
    """Bin discrete-integer columns into B equipopulation bins."""
    def __init__(self, B=32):
        self.B = B; self.edges = {}

    def fit(self, df, cols):
        for c in cols:
            v = df[c].values.astype(float)
            qs = np.linspace(0, 100, self.B+1)
            edges = np.unique(np.percentile(v, qs))
            if len(edges) < 2:
                raise ValueError(f"BinTransformer: column '{c}' has only one unique value — cannot bin.")
            if len(edges) < self.B + 1:
                import warnings
                warnings.warn(f"BinTransformer: column '{c}' has only {len(edges)-1} unique quantile edges "
                              f"(requested {self.B} bins). Inverse may be approximate.")
            self.edges[c] = edges
        return self

    def transform(self, df, cols):
        parts = []
        self.col_slices = {}
        offset = 0
        for c in cols:
            v    = df[c].values.astype(float)
            bins = np.digitize(v, self.edges[c][1:-1])  # 0..B-1
            bins = np.clip(bins, 0, self.B-1)
            oh   = np.zeros((len(v), self.B), dtype=np.float32)
            oh[np.arange(len(v)), bins] = 1.0
            parts.append(oh)
            self.col_slices[c] = (offset, offset+self.B)
            offset += self.B
        return np.column_stack(parts) if parts else np.zeros((len(df),0))

    def inverse(self, arr, cols):
        out = {}
        for c in cols:
            s, e = self.col_slices[c]
            bins  = np.argmax(arr[:, s:e], axis=1)
            edges = self.edges[c]
            centroids = [(edges[i]+edges[i+1])/2 if i+1<len(edges) else edges[-1]
                         for i in range(min(self.B, len(edges)-1))]
            out[c] = np.array([centroids[min(b, len(centroids)-1)] for b in bins])
        return out


# ── Quantile continuous transformer (replaces VGM for continuous cols) ──
class QuantileContTransformer:
    """Per-column QuantileTransformer(uniform) for continuous features.
    Emits ONE scalar per column in [-1,1] (compatible with the existing tanh head).
    Replaces VGM, which floored KS ~0.17 by smearing point masses (e.g. DURATION_OUT
    is 44.7% zeros): VGM/binning round-trip those cols at KS 0.45/0.26, a quantile
    transform at ~0.001. width=1 col per feature (vs VGM's K+1)."""
    def __init__(self, n_quantiles=1000, subsample=200_000):
        from sklearn.preprocessing import QuantileTransformer
        self._QT = QuantileTransformer
        self.n_quantiles = n_quantiles
        self.subsample = subsample
        self.qt = {}
        self.width = 1           # output columns per continuous feature
        self.col_slices = {}

    def fit(self, df, cols):
        for c in cols:
            n = int(min(self.n_quantiles, max(10, df[c].notna().sum())))
            qt = self._QT(n_quantiles=n, output_distribution='uniform',
                          subsample=self.subsample, random_state=SEED)
            qt.fit(df[[c]].values.astype(float))
            self.qt[c] = qt
        return self

    def transform(self, df, cols):
        self.col_slices = {}
        parts = []
        for i, c in enumerate(cols):
            v = self.qt[c].transform(df[[c]].values.astype(float))   # [0,1]
            parts.append((v * 2.0 - 1.0).astype(np.float32))         # -> [-1,1]
            self.col_slices[c] = (i, i + 1)
        return np.column_stack(parts) if parts else np.zeros((len(df), 0), np.float32)

    def inverse(self, arr, cols):
        out = {}
        for c in cols:
            s, _ = self.col_slices[c]
            q01 = np.clip((arr[:, s] + 1.0) / 2.0, 0.0, 1.0)
            out[c] = self.qt[c].inverse_transform(q01.reshape(-1, 1)).ravel()
        return out


# ── CIF-32 Preprocessor (combines all transformers) ───────────
class CIF32Preprocessor:
    def __init__(self, K=5, B=32):
        self.vgm = QuantileContTransformer()
        self.bins = BinTransformer(B=B)
        self.cat_encoders = {c: LabelEncoder() for c in CAT_COLS}
        self.n_cat_classes = {}
        self.fitted = False
        self.output_info = []   # list of (dim, type_str) for generator output heads

    def fit(self, df):
        print('Fitting quantile transformers...')
        self.vgm.fit(df, CONT_COLS)
        print('Fitting bin transformers...')
        self.bins.fit(df, INT_COLS)
        for c in CAT_COLS:
            self.cat_encoders[c].fit(df[c].astype(int).astype(str))
            self.n_cat_classes[c] = len(self.cat_encoders[c].classes_)
        self.fitted = True
        # build output_info in same column order as transform()
        self.output_info = []
        for c in CAT_COLS:
            self.output_info.append((self.n_cat_classes[c], 'softmax'))
        for c in CONT_COLS:
            self.output_info.append((1, 'tanh'))     # quantile value in [-1,1] (1 scalar/col)
        for c in INT_COLS:
            self.output_info.append((self.bins.B, 'softmax'))
        for c in BIN_COLS:
            self.output_info.append((2, 'softmax'))  # binary → 2-class
        self.output_info.append((2, 'softmax'))      # missingness
        self.out_dim = sum(d for d,_ in self.output_info)
        print(f'Preprocessor fitted. Output dim = {self.out_dim}')
        return self

    def transform(self, df):
        parts = []
        # categorical (one-hot)
        for c in CAT_COLS:
            v = df[c].astype(int).astype(str)
            enc = self.cat_encoders[c].transform(v.map(lambda x: x if x in self.cat_encoders[c].classes_ else self.cat_encoders[c].classes_[0]))
            nc = self.n_cat_classes[c]
            oh = np.zeros((len(df), nc), dtype=np.float32)
            oh[np.arange(len(df)), enc] = 1.0
            parts.append(oh)
        # continuous (VGM)
        parts.append(self.vgm.transform(df, CONT_COLS).astype(np.float32))
        # discrete-int (bins)
        parts.append(self.bins.transform(df, INT_COLS).astype(np.float32))
        # binary (2-class one-hot)
        for c in BIN_COLS:
            v = df[c].astype(int).clip(0,1).values
            oh = np.zeros((len(df), 2), dtype=np.float32)
            oh[np.arange(len(df)), v] = 1.0
            parts.append(oh)
        # missingness
        v = df['missingness'].astype(int).clip(0,1).values
        oh = np.zeros((len(df), 2), dtype=np.float32)
        oh[np.arange(len(df)), v] = 1.0
        parts.append(oh)
        return np.concatenate(parts, axis=1)

print('Preprocessor classes defined.')

Preprocessor classes defined.


In [8]:
# ── store inf-clip statistics inside the preprocessor so transform() applies
#    them consistently to BOTH train and test data ────────────────────────────
class _InfClipper:
    """Learned 99th-percentile caps for inf replacement — applied in transform()."""
    def __init__(self): self.caps = {}; self.medians = {}
    def fit(self, df, cols):
        for c in cols:
            s = df[c].replace([np.inf, -np.inf], np.nan)
            finite = s[np.isfinite(s)]
            self.caps[c]    = finite.quantile(0.99)
            self.medians[c] = finite.median()
        return self
    def apply(self, df, cols):
        df = df.copy()
        for c in cols:
            s = df[c].replace([np.inf, -np.inf], np.nan)
            s = s.fillna(self.medians[c]).clip(lower=0, upper=self.caps[c])
            df[c] = s
        return df

inf_clipper = _InfClipper().fit(train_df, CONT_COLS)
train_df = inf_clipper.apply(train_df, CONT_COLS)
test_df  = inf_clipper.apply(test_df,  CONT_COLS)   # same caps → no leakage
print('Inf/NaN handled in train and test using training-data statistics.')

# ── fit preprocessor on training data only ─────────────────────
prep = CIF32Preprocessor(K=CFG['K_VGM'], B=CFG['N_BINS'])
prep.fit(train_df)

print('Transforming train/test sets...')
X_train = prep.transform(train_df)
X_test  = prep.transform(test_df)
y_train = train_df['class_id'].values
y_test  = test_df['class_id'].values
src_train = train_df['source_id'].values
print(f'X_train shape: {X_train.shape}  |  X_test shape: {X_test.shape}')

Inf/NaN handled in train and test using training-data statistics.
Fitting quantile transformers...
Fitting bin transformers...
Preprocessor fitted. Output dim = 294
Transforming train/test sets...
X_train shape: (2639567, 294)  |  X_test shape: (1319784, 294)


## 3 · NSG-IDS Model Architecture

In [9]:
# ── building blocks ────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, dim, use_bn=True, dropout=0.1):
        super().__init__()
        layers = [nn.Linear(dim, dim)]
        if use_bn: layers.append(nn.BatchNorm1d(dim))
        layers += [nn.LeakyReLU(0.2), nn.Dropout(dropout), nn.Linear(dim, dim)]
        if use_bn: layers.append(nn.BatchNorm1d(dim))
        layers.append(nn.LeakyReLU(0.2))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return x + self.net(x)


class SelfAttnModule(nn.Module):
    """Reshape → multi-head self-attention over 32 CIF feature slots → reshape back.
    Uses manual Q/K/V projections instead of nn.MultiheadAttention to avoid
    a silent-NaN bug in nn.MultiheadAttention on MPS (Apple Silicon)."""
    def __init__(self, n_feat=32, feat_dim=32, n_heads=8):
        super().__init__()
        self.n_feat   = n_feat
        self.d        = feat_dim
        self.n_heads  = n_heads
        self.head_dim = feat_dim // n_heads
        self.scale    = self.head_dim ** -0.5
        self.q_proj   = nn.Linear(feat_dim, feat_dim)
        self.k_proj   = nn.Linear(feat_dim, feat_dim)
        self.v_proj   = nn.Linear(feat_dim, feat_dim)
        self.out_proj = nn.Linear(feat_dim, feat_dim)
        self.norm1    = nn.LayerNorm(feat_dim)
        self.ffn      = nn.Sequential(nn.Linear(feat_dim, feat_dim*2), nn.GELU(),
                                      nn.Linear(feat_dim*2, feat_dim))
        self.norm2    = nn.LayerNorm(feat_dim)

    def forward(self, h):  # h: (B, n_feat*d)
        B = h.shape[0]
        H = h.view(B, self.n_feat, self.d)                             # (B, 32, d)
        Q = self.q_proj(H).view(B, self.n_feat, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(H).view(B, self.n_feat, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(H).view(B, self.n_feat, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (Q @ K.transpose(-2, -1)) * self.scale                 # (B, H, 32, 32)
        a = (attn.softmax(dim=-1) @ V).transpose(1, 2).contiguous().view(B, self.n_feat, self.d)
        a = self.out_proj(a)
        H = self.norm1(H + a)
        H = self.norm2(H + self.ffn(H))
        return H.reshape(B, -1)                                        # (B, 32*d)


class DiscreteAwareHead(nn.Module):
    """Per-CIF-feature output heads applying type-specific activation."""
    def __init__(self, in_dim, output_info):
        super().__init__()
        self.info  = output_info
        self.heads = nn.ModuleList([nn.Linear(in_dim, sz) for sz, _ in output_info])

    def forward(self, h, tau=1.0):
        outs = []
        for head, (sz, typ) in zip(self.heads, self.info):
            logits = head(h)
            if typ == 'softmax':
                logits = logits.clamp(-15, 15)
                # Always use the manual Gumbel path: F.gumbel_softmax on MPS can
                # return silent NaN (no RuntimeError) due to subnormals in exponential_().
                U = torch.zeros_like(logits).uniform_().clamp(1e-6, 1 - 1e-6)
                gumbels = (-torch.log(-torch.log(U)) + logits) / tau
                out = gumbels.softmax(dim=-1)
                if not torch.isfinite(out).all():
                    out = (logits / tau).softmax(dim=-1)  # safe fallback
                outs.append(out)
            else:  # tanh
                outs.append(torch.tanh(logits))
        return torch.cat(outs, dim=-1)


# ── Generator ─────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, z_dim, embed_dim, hidden_dim, n_blocks, output_info,
                 n_classes, n_sources, n_feat=32, feat_dim=32, n_heads=8,
                 dropout=0.1, use_attn=True, use_da=True):
        super().__init__()
        self.use_attn = use_attn
        self.use_da   = use_da
        self.n_feat   = n_feat
        self.feat_dim = feat_dim
        inp_dim = z_dim + embed_dim
        out_dim = sum(sz for sz,_ in output_info)

        # class-source embedding
        self.class_emb  = nn.Embedding(n_classes, embed_dim // 2)
        self.source_emb = nn.Embedding(n_sources, embed_dim // 2)

        # projection + residual
        self.proj   = nn.Linear(inp_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResBlock(hidden_dim, use_bn=True, dropout=dropout)
                                     for _ in range(n_blocks)])
        # attention bridge: hidden_dim → n_feat*feat_dim
        attn_dim = n_feat * feat_dim
        self.to_attn   = nn.Linear(hidden_dim, attn_dim)
        if use_attn:
            self.attn_mod  = SelfAttnModule(n_feat, feat_dim, n_heads)
        self.from_attn = nn.Linear(attn_dim, hidden_dim)

        # output
        if use_da:
            self.out_head = DiscreteAwareHead(hidden_dim, output_info)
        else:
            self.out_head = nn.Sequential(nn.Linear(hidden_dim, out_dim), nn.Tanh())

    def forward(self, z, class_ids, source_ids, tau=1.0):
        e = torch.cat([self.class_emb(class_ids), self.source_emb(source_ids)], dim=-1)
        h = F.leaky_relu(self.proj(torch.cat([z, e], dim=-1)), 0.2)
        for blk in self.blocks: h = blk(h)
        a = self.to_attn(h)
        if self.use_attn: a = self.attn_mod(a)
        h = h + F.leaky_relu(self.from_attn(a), 0.2)
        if self.use_da:
            return self.out_head(h, tau)
        return self.out_head(h)


# ── Critic (Wasserstein) ───────────────────────────────────────
class Critic(nn.Module):
    def __init__(self, in_dim, embed_dim, hidden_dim, n_blocks,
                 n_classes, n_sources, n_feat=32, feat_dim=32,
                 n_heads=8, dropout=0.1):
        super().__init__()
        self.class_emb  = nn.Embedding(n_classes, embed_dim // 2)
        self.source_emb = nn.Embedding(n_sources, embed_dim // 2)
        inp = in_dim + embed_dim
        self.proj  = nn.Linear(inp, hidden_dim)
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.LeakyReLU(0.2), nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.LeakyReLU(0.2)
            ) for _ in range(n_blocks)
        ])
        attn_dim = n_feat * feat_dim
        self.to_attn   = nn.Linear(hidden_dim, attn_dim)
        self.attn_mod  = SelfAttnModule(n_feat, feat_dim, n_heads)
        self.from_attn = nn.Linear(attn_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, class_ids, source_ids):
        e = torch.cat([self.class_emb(class_ids), self.source_emb(source_ids)], dim=-1)
        h = F.leaky_relu(self.proj(torch.cat([x, e], dim=-1)), 0.2)
        for blk in self.blocks: h = h + blk(h)
        a = self.to_attn(h)
        a = self.attn_mod(a)
        h = h + F.leaky_relu(self.from_attn(a), 0.2)
        return self.out(h)

print('Model classes defined.')

Model classes defined.


## 4 · Training

In [10]:
# ── Dataset wrapper ────────────────────────────────────────────
class IDSDataset(Dataset):
    def __init__(self, X, y_class, y_source):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.yc = torch.tensor(y_class,  dtype=torch.long)
        self.ys = torch.tensor(y_source, dtype=torch.long)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.yc[i], self.ys[i]

# ── Source-balanced sampler: exactly equal source probability ──────────────
# For source s with classes C_s: w_{c,s} = 1 / (N_src × sqrt(n_{c,s}) × Σ_c' sqrt(n_{c',s}))
# This guarantees Σ_{samples in s} w_i = 1/N_src exactly, so every source
# contributes equally regardless of its total row count.
def make_weighted_sampler(y_class, y_source):
    cs = list(zip(y_class, y_source))
    pair_counts  = defaultdict(int)
    for c, s in cs: pair_counts[(c, s)] += 1
    n_sources    = len(set(s for _, s in cs))
    src_sqrt_sum = defaultdict(float)
    for (c, s), cnt in pair_counts.items(): src_sqrt_sum[s] += cnt**0.5
    weights = np.array([
        1.0 / (n_sources * (pair_counts[(c, s)]**0.5 + 1e-6) * src_sqrt_sum[s])
        for c, s in cs
    ])
    weights /= weights.sum()
    return torch.utils.data.WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.float32),
        num_samples=len(weights), replacement=True)

# ── Gradient penalty ──────────────────────────────────────────
def gradient_penalty(critic, real, fake, class_ids, source_ids, device):
    B = real.shape[0]
    eps = torch.rand(B, 1, device=device)
    hat = (eps * real + (1-eps) * fake).requires_grad_(True)
    score = critic(hat, class_ids, source_ids)
    grad  = torch.autograd.grad(score, hat,
                                grad_outputs=torch.ones_like(score),
                                create_graph=True)[0]
    gp = ((torch.sqrt(torch.sum(grad ** 2, dim=1) + 1e-12) - 1) ** 2).mean()
    return gp

# ── Training function ─────────────────────────────────────────
def train_nsgids(X, y_class, y_source, cfg, variant='full', n_epochs=None,
                 eval_every=0, eval_fn=None):
    """
    variant: 'full' | 'no_attn' | 'no_da' | 'no_gp' | 'no_cond'
    Returns (generator, critic, history)
    """
    use_attn = variant != 'no_attn'
    use_da   = variant != 'no_da'
    use_gp   = variant != 'no_gp'
    use_cond = variant != 'no_cond'
    epochs   = n_epochs or (cfg['FAST_EPOCHS'] if cfg['FAST_EPOCHS'] > 0 else cfg['N_EPOCHS'])

    # ----- on-GPU data pipeline -----------------------------------
    # Keep the whole (tabular) dataset resident on the accelerator and draw
    # weighted batches with torch.multinomial on-device. Removes the per-batch
    # WeightedRandomSampler + host->GPU copy that starves the GPU on this small
    # model (no DataLoader / workers needed). VRAM: X is rows*out_dim*4 B; if it
    # OOMs at DATA_PCT=1.0 on 6 GB, lower DATA_PCT. multinomial needs rows<=2**24.
    X_dev  = torch.as_tensor(X, dtype=torch.float32).to(DEVICE)
    yc_dev = torch.as_tensor(y_class,  dtype=torch.long).to(DEVICE)
    ys_dev = torch.as_tensor(y_source, dtype=torch.long).to(DEVICE)
    # reuse the exact source-balanced weighting, just move it on-device
    sample_w = make_weighted_sampler(y_class, y_source).weights.to(DEVICE, dtype=torch.float32)
    steps_per_epoch = len(X_dev) // cfg['BATCH_SIZE']  # == DataLoader drop_last batch count

    G = Generator(cfg['Z_DIM'], cfg['EMBED_DIM'], cfg['HIDDEN_DIM'],
                  cfg['N_RES_BLOCKS'], prep.output_info,
                  N_CLASSES, N_SOURCES,
                  n_feat=32, feat_dim=cfg['ATTN_FEAT_DIM'],
                  n_heads=cfg['ATTN_HEADS'], dropout=cfg['DROPOUT'],
                  use_attn=use_attn, use_da=use_da).to(DEVICE)

    C = Critic(prep.out_dim, cfg['EMBED_DIM'], cfg['HIDDEN_DIM'],
               cfg['N_RES_BLOCKS'], N_CLASSES, N_SOURCES,
               n_feat=32, feat_dim=cfg['ATTN_FEAT_DIM'],
               n_heads=cfg['ATTN_HEADS'], dropout=cfg['DROPOUT']).to(DEVICE)

    optG = torch.optim.Adam(G.parameters(), lr=cfg['LR'], betas=cfg['BETAS'])
    optC = torch.optim.Adam(C.parameters(), lr=cfg['LR'], betas=cfg['BETAS'])

    # ── EMA generator: averaged weights give smoother, higher-fidelity samples ──
    ema_decay = cfg.get('EMA_DECAY', 0.0)
    use_ema   = bool(ema_decay and ema_decay > 0)
    if use_ema:
        import copy
        G_ema = copy.deepcopy(G).eval()
        for p in G_ema.parameters(): p.requires_grad_(False)
    G_eval    = G_ema if use_ema else G
    ema_step  = 0
    c_step    = 0          # global critic-update counter for lazy GP
    gp_every  = max(1, int(cfg.get('GP_EVERY', 1)))
    fm_weight = cfg.get('FM_WEIGHT', 0.0)
    if fm_weight > 0:  # continuous VGM block bounds (mode one-hots + values)
        _catdim = sum(prep.n_cat_classes[c] for c in CAT_COLS)
        fm_lo, fm_hi = _catdim, _catdim + len(CONT_COLS) * prep.vgm.width

    use_amp = cfg.get('USE_AMP', False) and DEVICE == 'cuda'
    # bf16 keeps fp32 exponent range -> stable WGAN-GP grad-norm and needs no
    # GradScaler (no fp16 underflow). Scalers stay disabled so the scale/
    # unscale/step calls below act as transparent pass-throughs.
    amp_dtype = torch.bfloat16
    scaler_G = torch.amp.GradScaler('cuda', enabled=False)
    scaler_C = torch.amp.GradScaler('cuda', enabled=False)
    amp_ctx  = torch.amp.autocast(device_type='cuda', dtype=amp_dtype, enabled=use_amp)

    history = {'G_loss': [], 'C_loss': [], 'GP': [], 'fidelity': []}
    tau_decay = (cfg['TAU_END'] / cfg['TAU_START']) ** (1.0 / epochs)
    tau = cfg['TAU_START']

    _pct_used = cfg.get('DATA_PCT', 1.0) * 100
    print('[train_nsgids] variant=' + str(variant) +
          f' | epochs={epochs} | data={_pct_used:.0f}% | device={DEVICE}', flush=True)
    epoch_bar = tqdm(range(1, epochs + 1), desc=f'Training ({variant})', leave=True, dynamic_ncols=True)
    for epoch in epoch_bar:
        G.train(); C.train()
        # running sums — avoids per-step list allocation and np.mean in tight loop.
        # MPS: every .item() forces a GPU→CPU sync (~6×/batch) that stalls the
        # pipeline; accumulate on-device and sync ONCE per epoch. CUDA also
        # pays a sync per .item(), so we defer on both accelerators.
        _defer_sync = (DEVICE in ('mps', 'cuda'))
        if _defer_sync:
            g_sum  = torch.zeros((), device=DEVICE)
            c_sum  = torch.zeros((), device=DEVICE)
            gp_sum = torch.zeros((), device=DEVICE)
        else:
            g_sum = c_sum = gp_sum = 0.0
        n_batches = 0

        for _step in range(steps_per_epoch):
            # weighted batch indices drawn on-device (no host copy)
            idx    = torch.multinomial(sample_w, cfg['BATCH_SIZE'], replacement=True)
            real_x = X_dev[idx]
            real_c = yc_dev[idx]
            real_s = ys_dev[idx]
            B = real_x.shape[0]
            if not use_cond:
                real_c = torch.zeros_like(real_c)
                real_s = torch.zeros_like(real_s)

            # ─ critic steps ─
            bad_batch = False
            for _ in range(cfg['CRITIC_STEPS']):
                c_step += 1
                z = torch.randn(B, cfg['Z_DIM'], device=DEVICE)
                # forward under AMP (fp16 on CUDA, no-op on CPU/MPS)
                with amp_ctx:
                    fake_x = G(z, real_c, real_s, tau).detach()
                if not torch.isfinite(fake_x).all():
                    bad_batch = True
                    break
                # GP computed outside AMP to avoid scale issues with autograd.grad
                with amp_ctx:
                    loss_C = C(fake_x, real_c, real_s).mean() - C(real_x, real_c, real_s).mean()
                if use_gp and (c_step % gp_every == 0):
                    gp = gradient_penalty(C, real_x, fake_x, real_c, real_s, DEVICE)
                    if torch.isfinite(gp):
                        gp = gp.clamp(max=50.0)  # safety cap: prevents catastrophic loss spikes
                        loss_C = loss_C + cfg['LAMBDA_GP'] * gp
                        gp_sum = gp_sum + (gp.detach() if _defer_sync else gp.item())
                if not torch.isfinite(loss_C):
                    bad_batch = True
                    break
                optC.zero_grad()
                scaler_C.scale(loss_C).backward()
                scaler_C.unscale_(optC)   # unscale so clip sees real grads (no-op when AMP off)
                torch.nn.utils.clip_grad_norm_(C.parameters(), max_norm=1.0)
                scaler_C.step(optC)
                scaler_C.update()
                if any(not p.data.isfinite().all() for p in C.parameters()):
                    print(f'[warn] critic NaN at step {c_step} -> reinitialising C', flush=True)
                    C.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)
                    optC = torch.optim.Adam(C.parameters(), lr=cfg['LR'], betas=cfg['BETAS'])
                    bad_batch = True
                    break
                if not use_gp:            # WGAN weight clipping AFTER the optimizer step
                    for p in C.parameters(): p.data.clamp_(-0.01, 0.01)
                c_sum = c_sum + (loss_C.detach() if _defer_sync else loss_C.item())

            if bad_batch:  # skip generator update — NaN grads would corrupt weights
                continue

            # ─ generator step ─
            z = torch.randn(B, cfg['Z_DIM'], device=DEVICE)
            with amp_ctx:
                fake_x = G(z, real_c, real_s, tau)
                loss_G = -C(fake_x, real_c, real_s).mean()
                if fm_weight > 0:
                    # moment matching on the continuous VGM block only (mode+value):
                    # sharpens continuous marginals without disturbing categorical
                    # frequencies (which the critic + Gumbel path already handle).
                    rf = real_x[:, fm_lo:fm_hi]; ff = fake_x[:, fm_lo:fm_hi]
                    fm = (ff.mean(0) - rf.mean(0)).pow(2).mean() \
                       + (ff.std(0)  - rf.std(0)).pow(2).mean()
                    loss_G = loss_G + fm_weight * fm
            optG.zero_grad()
            scaler_G.scale(loss_G).backward()
            scaler_G.unscale_(optG)   # unscale before clipping (no-op when AMP off)
            torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
            scaler_G.step(optG)
            scaler_G.update()
            # ── update EMA weights (adaptive warmup so short runs benefit too) ──
            if use_ema:
                ema_step += 1
                d = min(ema_decay, (1 + ema_step) / (10 + ema_step))
                with torch.no_grad():
                    for pe, pl in zip(G_ema.parameters(), G.parameters()):
                        pe.mul_(d).add_(pl.detach(), alpha=1 - d)
                    for be, bl in zip(G_ema.buffers(), G.buffers()):
                        be.copy_(bl)
            g_sum = g_sum + (loss_G.detach() if _defer_sync else loss_G.item())
            n_batches += 1

        if n_batches == 0:
            # entire epoch produced non-finite samples (rare MPS init pathology):
            # re-initialise G so the run never silently stalls at NaN.
            print(f'[warn] epoch {epoch}: all batches non-finite -> reinitialising G', flush=True)
            G.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)
            optG = torch.optim.Adam(G.parameters(), lr=cfg['LR'], betas=cfg['BETAS'])
            if use_ema:
                G_ema = copy.deepcopy(G).eval()
                for p in G_ema.parameters(): p.requires_grad_(False)
                G_eval = G_ema
            tau = cfg['TAU_START']
            continue

        if _defer_sync:   # single GPU→CPU sync per epoch instead of ~6 per batch
            g_sum, c_sum, gp_sum = float(g_sum), float(c_sum), float(gp_sum)
        tau *= tau_decay
        g_mean = g_sum / max(n_batches, 1)
        c_mean = c_sum / max(n_batches * cfg['CRITIC_STEPS'], 1)
        gp_steps = max(n_batches // gp_every, 1)
        gp_mean = gp_sum / gp_steps if use_gp else 0.0
        history['G_loss'].append(g_mean)
        history['C_loss'].append(c_mean)
        history['GP'].append(gp_mean)

        epoch_bar.set_postfix(G=f'{g_mean:.4f}', C=f'{c_mean:.4f}',
                              GP=f'{gp_mean:.4f}', tau=f'{tau:.3f}')

        # ── periodic fidelity check so we don't wait for all epochs to see KPIs ──
        if eval_every and eval_fn is not None and (epoch % eval_every == 0 or epoch == epochs):
            m = eval_fn(G_eval); m['epoch'] = epoch
            history['fidelity'].append(m)
            print(f'[eval @ epoch {epoch:3d}] KS={m["KS"]:.4f}  '
                  f'CD={m["CD"]:.4f}  JSD={m["JSD"]:.4f}', flush=True)
            G.train(); C.train()

        if epoch % 50 == 0 or epoch == 1 or epoch == epochs:
            print(f'Epoch {epoch:3d}/{epochs} | G={g_mean:.4f} '
                  f'C={c_mean:.4f} τ={tau:.3f}', flush=True)

    return G_eval, C, history

print('Training utilities defined.')


Training utilities defined.


In [11]:
# ── Fast in-training fidelity probe (KS / CD / JSD) ────────────────────────────
# Self-contained so train_nsgids can call it before the full eval cell is defined.
# Decodes VGM/bin blocks back to raw feature space and compares a real test
# subsample against synthetic rows generated for the SAME (class, source) ids.
def _decode_cont_int(X_enc):
    cat_dim = sum(prep.n_cat_classes[c] for c in CAT_COLS)
    B = CFG['N_BINS']
    W = prep.vgm.width  # 1 col per continuous feature (quantile rep)
    vgm_len = len(CONT_COLS) * W
    prep.vgm.col_slices  = {c: (i*W, i*W+W) for i, c in enumerate(CONT_COLS)}
    cont = prep.vgm.inverse(X_enc[:, cat_dim:cat_dim+vgm_len], CONT_COLS)
    int_start = cat_dim + vgm_len
    prep.bins.col_slices = {c: (i*B, i*B+B) for i, c in enumerate(INT_COLS)}
    ints = prep.bins.inverse(X_enc[:, int_start:int_start+len(INT_COLS)*B], INT_COLS)
    return pd.DataFrame({**cont, **ints})

def _ks(rd, fd):
    s = [ks_2samp(rd[c].dropna(), fd[c].dropna())[0]
         for c in CONT_COLS if rd[c].notna().any() and fd[c].notna().any()]
    return float(np.mean(s)) if s else float('nan')

def _cd(rd, fd):
    rd = rd.replace([np.inf, -np.inf], np.nan); fd = fd.replace([np.inf, -np.inf], np.nan)
    cols = [c for c in CONT_COLS + INT_COLS if rd[c].nunique() > 1 and fd[c].nunique() > 1]
    if len(cols) < 2: return float('nan')
    R = np.clip(np.nan_to_num(rd[cols].corr().values), -1.0, 1.0)
    S = np.clip(np.nan_to_num(fd[cols].corr().values), -1.0, 1.0)
    return float(np.linalg.norm(R - S, 'fro') / len(cols))

def _jsd(Xr, Xf):
    js, off = [], 0
    for c in CAT_COLS:
        nc = prep.n_cat_classes[c]
        r = Xr[:, off:off+nc].mean(0) + 1e-8; r /= r.sum()
        f = Xf[:, off:off+nc].mean(0) + 1e-8; f /= f.sum()
        js.append(float(jensenshannon(r, f))); off += nc
    return float(np.mean(js)) if js else float('nan')

@torch.no_grad()
def quick_fidelity(G, n=2000):
    was = G.training; G.eval()
    sub = test_df.sample(min(n, len(test_df)), random_state=SEED)
    Xr = prep.transform(sub)
    ci = torch.tensor(sub['class_id'].values,  dtype=torch.long, device=DEVICE)
    si = torch.tensor(sub['source_id'].values, dtype=torch.long, device=DEVICE)
    z  = torch.randn(len(sub), CFG['Z_DIM'], device=DEVICE)
    Xf = G(z, ci, si, CFG['TAU_END']).cpu().numpy()
    if was: G.train()
    rd, fd = _decode_cont_int(Xr), _decode_cont_int(Xf)
    return dict(KS=_ks(rd, fd), CD=_cd(rd, fd), JSD=_jsd(Xr, Xf))

print("quick_fidelity() ready -> train_nsgids(..., eval_every=20, eval_fn=quick_fidelity)")


quick_fidelity() ready -> train_nsgids(..., eval_every=20, eval_fn=quick_fidelity)


In [ ]:
# ══════════════════════════════════════════════════════════════
#  RUN MAIN TRAINING  (NSG-IDS Fused)
#  Smoke vs full is controlled by SMOKE_TEST in the config cell.
#  eval_every uses SMOKE_EVAL_EVERY so the KS/CD/JSD probe prints
#  often during a smoke run (epochs 5/10/15/20) and you can SEE the
#  trend without waiting for the whole schedule.
# ══════════════════════════════════════════════════════════════
print('Training NSG-IDS (Fused)...')
_t0 = time.time()
G_fused, C_fused, hist_fused = train_nsgids(
    X_train, y_train, src_train, CFG, variant='full',
    eval_every=globals().get('SMOKE_EVAL_EVERY', 20), eval_fn=quick_fidelity)
_elapsed = time.time() - _t0
_ep = CFG['FAST_EPOCHS'] if CFG['FAST_EPOCHS'] > 0 else CFG['N_EPOCHS']
_per_ep = _elapsed / max(_ep, 1)
print(f'Training wall-clock: {_elapsed/60:.1f} min  ({_per_ep:.1f}s/epoch '
      f'at DATA_PCT={CFG.get("DATA_PCT",1.0)*100:.0f}%)')
# Rough full-run extrapolation (scales ~linearly with epochs and data fraction)
if globals().get('SMOKE_TEST', False):
    _full_ep, _full_pct = 300, 1.0
    _scale = (_full_ep / max(_ep, 1)) * (_full_pct / max(CFG.get('DATA_PCT', 1.0), 1e-9))
    print(f'  ~Extrapolated full run (300 ep, 100% data): '
          f'{_elapsed*_scale/3600:.1f} h for THIS cell alone '
          f'(ablation + baselines add more).')

torch.save(G_fused.state_dict(), './models/G_fused.pt')
print('Model saved.')


Training NSG-IDS (Fused)...
[train_nsgids] variant=full | epochs=100 | data=50% | device=cuda


Training (full):   0%|          | 0/100 [28:38<?, ?it/s, C=-1.3541, G=10.2754, GP=2.2252, tau=0.984]

Epoch   1/100 | G=10.2754 C=-1.3541 τ=0.984


Training (full):  18%|█▊        | 18/100 [8:41:26<40:34:56, 1781.66s/it, C=-0.5341, G=7.0655, GP=1.3068, tau=0.748]

## 5 · Synthetic Data Generation

In [ ]:
@torch.no_grad()
def generate_synthetic(G, n_per_class, label_enc, source_enc,
                        class_to_source=None, batch=4096, tau=0.2):
    """
    Generate n_per_class samples per class.
    Batches ALL classes together into large GPU passes instead of looping
    class-by-class, which maximises GPU utilisation and reduces kernel launch overhead.
    class_to_source: dict mapping class_id → preferred source_id
    """
    G.eval()
    if class_to_source is None:
        class_to_source = {c: c % N_SOURCES for c in range(N_CLASSES)}

    # Build index arrays for ALL samples upfront (no Python loop per sample)
    all_cls_ids = np.repeat(np.arange(N_CLASSES), n_per_class)      # (N_CLASSES * n_per_class,)
    all_src_ids = np.array([class_to_source[c] for c in all_cls_ids])
    total = len(all_cls_ids)

    use_amp = CFG.get('USE_AMP', False) and DEVICE == 'cuda'
    amp_ctx = torch.amp.autocast(device_type='cuda', enabled=use_amp)

    all_X = []
    for start in range(0, total, batch):
        end = min(start + batch, total)
        bs  = end - start
        z   = torch.randn(bs, CFG['Z_DIM'], device=DEVICE)
        ci  = torch.tensor(all_cls_ids[start:end], dtype=torch.long, device=DEVICE)
        si  = torch.tensor(all_src_ids[start:end], dtype=torch.long, device=DEVICE)
        with amp_ctx:
            x = G(z, ci, si, tau)
        all_X.append(x.cpu().numpy())

    X_syn = np.concatenate(all_X)
    y_syn = all_cls_ids
    print(f'Generated {len(X_syn):,} synthetic samples across {N_CLASSES} classes '
          f'({total // batch + 1} GPU batches of up to {batch})')
    return X_syn, y_syn


# Build class→source mapping from training distribution
from collections import Counter
cs_counts = Counter(zip(train_df['class_id'], train_df['source_id']))
class_to_src = {}
for cid in range(N_CLASSES):
    # pick source with most samples for this class
    best = max(range(N_SOURCES), key=lambda s: cs_counts.get((cid,s), 0))
    class_to_src[cid] = best

X_syn, y_syn = generate_synthetic(
    G_fused, CFG['N_SYNTH_PER_CLASS'], label_enc, source_enc, class_to_src)
np.save('./outputs/X_syn.npy', X_syn)
np.save('./outputs/y_syn.npy', y_syn)
print('Synthetic data saved.')

## 6 · Evaluation — Statistical Fidelity (Table 3)

In [ ]:
# ── decode preprocessed arrays back to raw feature space ──────
def decode_to_raw(X_enc):
    """
    Approximate inverse of CIF32Preprocessor.transform().
    Returns a DataFrame with continuous columns decoded.
    """
    # reconstruct offset for continuous VGM block
    cat_dim = sum(prep.n_cat_classes[c] for c in CAT_COLS)
    W = prep.vgm.width  # 1 col per continuous feature (quantile rep)
    vgm_block = X_enc[:, cat_dim: cat_dim + len(CONT_COLS)*W]
    # rebuild col_slices for the continuous block
    slices, offset = {}, 0
    for c in CONT_COLS:
        slices[c] = (offset, offset + W); offset += W
    prep.vgm.col_slices = slices
    cont_vals = prep.vgm.inverse(vgm_block, CONT_COLS)

    # int bin block
    int_start = cat_dim + len(CONT_COLS)*W
    int_block = X_enc[:, int_start: int_start + len(INT_COLS)*CFG['N_BINS']]
    islices, ioffset = {}, 0
    for c in INT_COLS:
        islices[c] = (ioffset, ioffset + CFG['N_BINS']); ioffset += CFG['N_BINS']
    prep.bins.col_slices = islices
    int_vals = prep.bins.inverse(int_block, INT_COLS)

    df = pd.DataFrame({**cont_vals, **int_vals})
    return df.replace([np.inf, -np.inf], np.nan)


src_names = {0:'CIC-IDS-2017', 1:'UNSW-NB15', 2:'NF-ToN-IoT'}

# ── KS statistic ──────────────────────────────────────────────
def mean_ks(real_enc, fake_enc):
    """Mean KS statistic across decoded continuous features."""
    real_df = decode_to_raw(real_enc)
    fake_df = decode_to_raw(fake_enc)
    stats = []
    for c in CONT_COLS:
        if c in real_df and c in fake_df:
            r, f = real_df[c].dropna(), fake_df[c].dropna()
            if len(r) > 0 and len(f) > 0:
                ks, _ = ks_2samp(r, f)
                stats.append(ks)
    return float(np.mean(stats)) if stats else float('nan')


# ── Correlation distance ───────────────────────────────────────
def corr_distance(real_enc, fake_enc):
    """Frobenius norm of Pearson correlation matrix difference on decoded features, normalised by n_features."""
    real_df = decode_to_raw(real_enc).replace([np.inf, -np.inf], np.nan)
    fake_df = decode_to_raw(fake_enc).replace([np.inf, -np.inf], np.nan)
    cols = [c for c in CONT_COLS + INT_COLS if c in real_df.columns and c in fake_df.columns]
    # drop columns constant in either set: corr is undefined there and the raw
    # (pre-fillna) values can blow the Frobenius norm past its valid <=2 range.
    cols = [c for c in cols if real_df[c].nunique() > 1 and fake_df[c].nunique() > 1]
    if len(cols) < 2:
        return float('nan')
    R = np.clip(np.nan_to_num(real_df[cols].corr().values), -1.0, 1.0)
    S = np.clip(np.nan_to_num(fake_df[cols].corr().values), -1.0, 1.0)
    return float(np.linalg.norm(R - S, 'fro') / len(cols))


# ── JSD on categorical columns ─────────────────────────────────
def mean_jsd(real_enc, fake_enc):
    """Mean JSD across one-hot categorical blocks."""
    jsds = []
    offset = 0
    for c in CAT_COLS:
        nc = prep.n_cat_classes[c]
        r  = real_enc[:, offset:offset+nc].mean(0) + 1e-8
        s  = fake_enc[:, offset:offset+nc].mean(0) + 1e-8
        r /= r.sum(); s /= s.sum()
        jsds.append(float(jensenshannon(r, s)))
        offset += nc
    return float(np.mean(jsds)) if jsds else float('nan')


# ── Run fidelity evaluation per source ────────────────────────
def eval_fidelity(G, source_ids_to_eval=None):
    results = {}
    srcs = source_ids_to_eval or test_df['source'].unique()
    G.eval()
    for src in srcs:
        src_test = test_df[test_df['source'] == src]
        if len(src_test) < 100:
            print(f'  Skipping source {src} ({src_names.get(src, src)}): '
                  f'only {len(src_test)} test samples (minimum 100 required)')
            continue
        print(f'  Evaluating source {src} ({src_names.get(src, src)}): {len(src_test):,} samples')
        X_real = prep.transform(src_test)

        # Generate synthetic with the SAME per-class proportions as the real
        # source split (paired conditioning). Generating a balanced synthetic
        # mixture and comparing it against the BENIGN-dominated real marginal
        # inflates KS/CD/JSD by construction; pairing the class mix makes the
        # comparison like-for-like.
        src_id  = int(src)
        cls_vec = src_test['class_id'].values
        MAX_EVAL = 20000  # cap large sources for speed, preserving class proportions
        if len(cls_vec) > MAX_EVAL:
            _rng = np.random.default_rng(SEED)
            _keep = _rng.choice(len(cls_vec), MAX_EVAL, replace=False)
            cls_vec = cls_vec[_keep]
            X_real  = X_real[_keep]
        G_parts = []
        for _start in range(0, len(cls_vec), 4096):
            _chunk = cls_vec[_start:_start+4096]
            with torch.no_grad():
                z  = torch.randn(len(_chunk), CFG['Z_DIM'], device=DEVICE)
                ci = torch.tensor(_chunk, dtype=torch.long, device=DEVICE)
                si = torch.full((len(_chunk),), src_id, dtype=torch.long, device=DEVICE)
                xf = G(z, ci, si, 0.2).cpu().numpy()
            G_parts.append(xf)
        X_fake = np.concatenate(G_parts)

        n = min(len(X_real), len(X_fake))
        results[src] = dict(
            KS = mean_ks(X_real[:n], X_fake[:n]),
            CD = corr_distance(X_real[:n], X_fake[:n]),
            JSD= mean_jsd(X_real[:n], X_fake[:n]),
        )
    return results

print('Running fidelity evaluation...')
G_fused.eval()
fidelity = eval_fidelity(G_fused)
print('\n=== Table 3: Statistical Fidelity (NSG-IDS Fused) ===')
print(f'{"Source":15s}  {"KS":>8s}  {"CD":>8s}  {"JSD":>8s}')
for src, m in sorted(fidelity.items()):
    print(f'{src_names.get(src,str(src)):15s}  {m["KS"]:8.4f}  {m["CD"]:8.4f}  {m["JSD"]:8.4f}')

## 7 · Machine Learning Efficacy — TSTR (Table 4)

In [ ]:
def tstr_eval(X_syn, y_syn, test_sets_by_source, n_estimators=200):
    """
    Train Random Forest on synthetic; evaluate on each real test split.
    Returns dict: source → {F1, Prec, Rec, Acc}
    """
    print(f'  Training RF on {len(X_syn):,} synthetic samples...')
    clf = RandomForestClassifier(n_estimators=n_estimators, n_jobs=4,  # limit cores to avoid OOM
                                  random_state=SEED, class_weight='balanced')
    clf.fit(X_syn, y_syn)

    results = {}
    for src, src_df in test_sets_by_source.items():
        if len(src_df) < 50: continue
        X_r = prep.transform(src_df)
        y_r = src_df['class_id'].values
        yp  = clf.predict(X_r)
        results[src] = dict(
            F1   = f1_score(y_r, yp, average='macro', zero_division=0),
            Prec = precision_score(y_r, yp, average='macro', zero_division=0),
            Rec  = recall_score(y_r, yp, average='macro', zero_division=0),
            Acc  = accuracy_score(y_r, yp),
        )
    return results


print('Running TSTR evaluation (NSG-IDS Fused)...')
mle_fused = tstr_eval(X_syn, y_syn, test_by_src, n_estimators=CFG['RF_TREES'])

# Real-data baseline (train on real, test on real)
print('Running real-data RF baseline...')
mle_real  = tstr_eval(X_train, y_train, test_by_src, n_estimators=CFG['RF_TREES'])

print('\n=== Table 4: TSTR Macro-F1 per Test Set ===')
print(f'{"Method":22s}  {"Source":15s}  {"F1":>6s}  {"Prec":>6s}  {"Rec":>6s}  {"Acc":>6s}')
for src, m in sorted(mle_fused.items()):
    print(f'{"NSG-IDS (Fused)":22s}  {src_names.get(src,str(src)):15s}  '
          f'{m["F1"]:6.4f}  {m["Prec"]:6.4f}  {m["Rec"]:6.4f}  {m["Acc"]:6.4f}')
for src, m in sorted(mle_real.items()):
    print(f'{"Real-Data Baseline":22s}  {src_names.get(src,str(src)):15s}  '
          f'{m["F1"]:6.4f}  {m["Prec"]:6.4f}  {m["Rec"]:6.4f}  {m["Acc"]:6.4f}')

## 8 · Attack Coverage (Table 5)

In [ ]:
def coverage_score(y_syn, min_samples=500):
    counts = Counter(y_syn)
    covered = sum(1 for c in range(N_CLASSES) if counts.get(c, 0) >= min_samples)
    return covered, N_CLASSES, 100.0 * covered / N_CLASSES

cov_covered, cov_total, cov_pct = coverage_score(y_syn, CFG['COVERAGE_MIN'])
print(f'\n=== Table 5: Attack Coverage ===')
print(f'NSG-IDS (Fused): {cov_covered}/{cov_total} classes covered ({cov_pct:.1f}%)')
print(f'Min samples threshold: {CFG["COVERAGE_MIN"]}')

# Per-class counts
cnt = Counter(y_syn)
df_cov = pd.DataFrame([
    {'Class': label_enc.classes_[c], 'Count': cnt.get(c,0),
     'Covered': cnt.get(c,0) >= CFG['COVERAGE_MIN']}
    for c in range(N_CLASSES)
]).sort_values('Count', ascending=False)
print(df_cov.to_string(index=False))

## 9 Â· Ablation Study (Table 6)

In [ ]:
# ==================================================================
#  9 · ABLATION STUDY (Table 6)  — SKIPPED during SMOKE_TEST.
#  Trains 5 NSG-IDS variants; this is one of the big multi-day costs,
#  so a smoke run skips it and just defines empty results downstream.
# ==================================================================
if globals().get('SMOKE_TEST', False):
    X_train_abl, y_train_abl, src_train_abl = X_train, y_train, src_train
    ablation_results = {}
    print('SMOKE_TEST: skipping ablation study (5 trainings) — cell 26.')
else:
    # ==================================================================
    #  9 Â· ABLATION STUDY  (Table 6)
    #  Trains each NSG-IDS variant on a stratified subsample (for speed),
    #  then scores TSTR-F1 / KS / CD / Coverage to isolate the contribution
    #  of attention, discrete-aware heads, gradient penalty and conditioning.
    #  Defines X_train_abl / y_train_abl / src_train_abl (reused by the
    #  baselines cell) and ablation_results (consumed by the final tables).
    # ==================================================================
    from sklearn.model_selection import train_test_split as _tts_abl

    # -- stratified subsample of the training set (keeps every class) --
    ABL_SUBSAMPLE = min(len(X_train), 60_000)
    if ABL_SUBSAMPLE < len(X_train):
        idx_abl, _ = _tts_abl(np.arange(len(X_train)), train_size=ABL_SUBSAMPLE,
                              stratify=y_train, random_state=SEED)
    else:
        idx_abl = np.arange(len(X_train))
    X_train_abl   = X_train[idx_abl]
    y_train_abl   = y_train[idx_abl]
    src_train_abl = src_train[idx_abl]
    print(f'Ablation subsample: {len(X_train_abl):,} rows')

    # -- shorter schedule for ablation runs --
    _full_epochs_abl = CFG['FAST_EPOCHS'] if CFG['FAST_EPOCHS'] > 0 else CFG['N_EPOCHS']
    ABL_EPOCHS = max(50, _full_epochs_abl // 6)

    ABL_VARIANTS = {
        'NSG-IDS (Full)':       'full',
        'w/o Self-Attention':   'no_attn',
        'w/o Discrete-Aware':   'no_da',
        'w/o Gradient Penalty': 'no_gp',
        'w/o Conditioning':     'no_cond',
    }

    def _mean_metric(vals):
        vals = [v for v in vals if v == v]  # drop NaN
        return float(np.mean(vals)) if vals else float('nan')

    ablation_results = {}
    for _name, _variant in ABL_VARIANTS.items():
        print(f'\n---- Ablation: {_name}  (variant={_variant}) ----')
        G_abl, _, _ = train_nsgids(X_train_abl, y_train_abl, src_train_abl, CFG,
                                   variant=_variant, n_epochs=ABL_EPOCHS)
        G_abl.eval()
        X_abl, y_abl = generate_synthetic(G_abl, min(2000, CFG['N_SYNTH_PER_CLASS']),
                                          label_enc, source_enc, class_to_src)
        fid = eval_fidelity(G_abl)
        mle = tstr_eval(X_abl, y_abl, test_by_src, n_estimators=50)
        cov = coverage_score(y_abl, CFG['COVERAGE_MIN'])[0]
        ablation_results[_name] = dict(
            F1       = _mean_metric([m['F1'] for m in mle.values()]),
            KS       = _mean_metric([m['KS'] for m in fid.values()]),
            CD       = _mean_metric([m['CD'] for m in fid.values()]),
            Coverage = int(cov),
        )
        r = ablation_results[_name]
        print(f'  {_name}: F1={r["F1"]:.4f}  KS={r["KS"]:.4f}  CD={r["CD"]:.4f}  cov={r["Coverage"]}')

    print('\n=== Table 6: Ablation ===')
    print(f'{"Configuration":<25} {"F1":>7} {"KS":>7} {"CD":>7} {"Coverage":>8}')
    for _name, _m in ablation_results.items():
        print(f'{_name:<25} {_m["F1"]:7.4f} {_m["KS"]:7.4f} {_m["CD"]:7.4f} {_m["Coverage"]:8d}')

## 10 · Baselines (SMOTE, CTGAN, Vanilla WGAN-GP)

In [ ]:
# ==================================================================
#  10 · BASELINES (SMOTE / Vanilla WGAN-GP)  — SKIPPED during SMOKE_TEST.
# ==================================================================
if globals().get('SMOKE_TEST', False):
    baseline_results = {}
    print('SMOKE_TEST: skipping baselines (SMOTE + Vanilla WGAN-GP) — cell 28.')
else:
    # Re-state ablation config so this cell can run independently of Cell 25
    _full_epochs = CFG['FAST_EPOCHS'] if CFG['FAST_EPOCHS'] > 0 else CFG['N_EPOCHS']
    ABL_EPOCHS   = max(50, _full_epochs // 6)
    cfg_abl      = dict(CFG)
    cfg_abl['CRITIC_STEPS'] = 1
    cfg_abl['BATCH_SIZE']   = min(CFG['BATCH_SIZE'], 128)

    from imblearn.over_sampling import SMOTE

    baseline_results = {}

    # ─ SMOTE (single source: CIC-IDS-2017 = source 0) ─
    print('Running SMOTE baseline...')
    src0_mask  = train_df['source'] == 0
    if src0_mask.sum() > 100:
        X_s0  = X_train[src0_mask]
        y_s0  = y_train[src0_mask]
        # only keep classes with >= 6 samples (SMOTE requirement)
        valid  = np.array([c for c in np.unique(y_s0) if (y_s0==c).sum() >= 6])
        mask2  = np.isin(y_s0, valid)
        try:
            sm    = SMOTE(k_neighbors=5, random_state=SEED)
            Xs_sm, ys_sm = sm.fit_resample(X_s0[mask2], y_s0[mask2])
            mle_smote = tstr_eval(Xs_sm, ys_sm, test_by_src, n_estimators=25)
            # SMOTE samples live in the SAME encoded CIF-32 space, so score them
            # directly against real source-0 test rows (do NOT reuse the NSG-IDS generator).
            _src0_test = test_df[test_df['source'] == 0]
            fid_smote = {}
            if len(_src0_test) >= 100:
                _Xr0  = prep.transform(_src0_test)
                _perm = np.random.default_rng(SEED).permutation(len(Xs_sm))
                _Xsm  = Xs_sm[_perm]
                _n    = min(len(_Xr0), len(_Xsm))
                fid_smote[0] = dict(
                    KS = mean_ks(_Xr0[:_n], _Xsm[:_n]),
                    CD = corr_distance(_Xr0[:_n], _Xsm[:_n]),
                    JSD= mean_jsd(_Xr0[:_n], _Xsm[:_n]),
                )
            cov_smote = coverage_score(ys_sm, CFG['COVERAGE_MIN'])
            baseline_results['SMOTE'] = dict(
                mle=mle_smote, fid=fid_smote, cov=cov_smote[0])
            print(f'  SMOTE: coverage={cov_smote[0]}')
        except Exception as e:
            print(f'  SMOTE failed: {e}')
    else:
        print('  Skipping SMOTE: insufficient CIC-IDS-2017 samples')

    # ─ Vanilla WGAN-GP (no attn, no discrete-aware) ─
    print('Running Vanilla WGAN-GP baseline...')
    G_van, _, _ = train_nsgids(X_train_abl, y_train_abl, src_train_abl, cfg_abl,
                                variant='no_attn', n_epochs=ABL_EPOCHS)
    G_van.eval()
    X_van, y_van = generate_synthetic(G_van, min(1000, CFG['N_SYNTH_PER_CLASS']),
                                       label_enc, source_enc, class_to_src)
    mle_van  = tstr_eval(X_van, y_van, test_by_src, n_estimators=25)
    fid_van  = eval_fidelity(G_van)
    cov_van  = coverage_score(y_van, CFG['COVERAGE_MIN'])
    baseline_results['WGAN-GP (Vanilla)'] = dict(mle=mle_van, fid=fid_van, cov=cov_van[0])
    print(f'  WGAN-GP Vanilla coverage: {cov_van[0]}')

## 11 · Visualisations

In [ ]:
# ── Training curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(hist_fused['G_loss'], label='Generator loss')
axes[0].set_title('Generator Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, alpha=0.3)
axes[1].plot(hist_fused['C_loss'], label='Critic loss', color='orange')
axes[1].set_title('Critic Loss'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)
axes[2].plot(hist_fused['GP'], label='Gradient Penalty', color='green')
axes[2].set_title('Gradient Penalty'); axes[2].set_xlabel('Epoch'); axes[2].grid(True, alpha=0.3)
plt.suptitle('NSG-IDS Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/training_curves.pdf', bbox_inches='tight')
plt.show(); print('Saved training_curves.pdf')

In [ ]:
# ── t-SNE: real vs synthetic ───────────────────────────────────
N_TSNE = 2000
rng = np.random.default_rng(SEED)
idx_r = rng.choice(len(X_test),  min(N_TSNE, len(X_test)),  replace=False)
idx_s = rng.choice(len(X_syn),   min(N_TSNE, len(X_syn)),   replace=False)
X_vis = np.concatenate([X_test[idx_r], X_syn[idx_s]])
labels_vis = np.concatenate([np.zeros(len(idx_r)), np.ones(len(idx_s))])
col_med = np.nanmedian(X_vis, axis=0)
nan_mask = np.isnan(X_vis)
X_vis = np.where(nan_mask, np.broadcast_to(col_med, X_vis.shape), X_vis)


print('Running t-SNE (this may take a minute)...')
from sklearn import __version__ as _skv
_mj, _mn = (int(x) for x in _skv.split('.')[:2])
_tsne_kw = {'max_iter': 500} if (_mj, _mn) >= (1, 5) else {'n_iter': 500}  # n_iter renamed in sklearn 1.5
emb = TSNE(n_components=2, perplexity=40, random_state=SEED, **_tsne_kw).fit_transform(X_vis)

fig, ax = plt.subplots(figsize=(8,6))
colors = ['#2196F3','#FF5722']
for i, (name, col) in enumerate([('Real', colors[0]), ('Synthetic', colors[1])]):
    mask = labels_vis == i
    ax.scatter(emb[mask,0], emb[mask,1], c=col, s=6, alpha=0.4, label=name)
ax.legend(fontsize=12); ax.set_title('t-SNE: Real vs NSG-IDS Synthetic', fontsize=13, fontweight='bold')
ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
plt.tight_layout()
plt.savefig('./figures/tsne_real_vs_syn.pdf', bbox_inches='tight')
plt.show(); print('Saved tsne_real_vs_syn.pdf')

In [ ]:
# ── Correlation heatmaps: real vs synthetic ────────────────────
N_FEAT_SHOW = min(15, len(CONT_COLS))  # show first N continuous features

cat_dim = sum(prep.n_cat_classes[c] for c in CAT_COLS)
vgm_len = len(CONT_COLS) * prep.vgm.width

def get_cont_block(X_enc):
    block = X_enc[:, cat_dim : cat_dim + vgm_len]
    # quantile rep: 1 scalar per continuous feature
    W = prep.vgm.width
    norm_vals = np.column_stack([block[:, k*W] for k in range(len(CONT_COLS))])
    return pd.DataFrame(norm_vals[:, :N_FEAT_SHOW],
                        columns=CONT_COLS[:N_FEAT_SHOW])

n_compare = min(3000, len(X_test), len(X_syn))
real_cont = get_cont_block(X_test[:n_compare])
syn_cont  = get_cont_block(X_syn[:n_compare])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(real_cont.corr(), ax=axes[0], cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            xticklabels=True, yticklabels=True, annot=False)
axes[0].set_title('Real Data Correlation', fontsize=12, fontweight='bold')
sns.heatmap(syn_cont.corr(), ax=axes[1], cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            xticklabels=True, yticklabels=True, annot=False)
axes[1].set_title('NSG-IDS Synthetic Correlation', fontsize=12, fontweight='bold')
plt.suptitle('Pearson Correlation Matrices (Continuous CIF-32 Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/correlation_heatmaps.pdf', bbox_inches='tight')
plt.show(); print('Saved correlation_heatmaps.pdf')

In [ ]:
# ── KS bar chart (per-source comparison) ──────────────────────
methods_ks = {}
for src in fidelity.keys():
    methods_ks.setdefault(src_names.get(src,str(src)), {})
    methods_ks[src_names.get(src,str(src))]['NSG-IDS (Fused)'] = fidelity[src]['KS']

if 'WGAN-GP (Vanilla)' in baseline_results:
    for src, m in baseline_results['WGAN-GP (Vanilla)']['fid'].items():
        sn = src_names.get(src,str(src))
        methods_ks.setdefault(sn, {})
        methods_ks[sn]['WGAN-GP (Vanilla)'] = m['KS']

srcs_list = list(methods_ks.keys())
all_methods = list(set(m for d in methods_ks.values() for m in d.keys()))
x = np.arange(len(srcs_list)); width = 0.35

fig, ax = plt.subplots(figsize=(9,5))
palette = ['#1565C0','#E65100','#2E7D32','#6A1B9A']
for i, method in enumerate(all_methods):
    vals = [methods_ks[s].get(method, np.nan) for s in srcs_list]
    ax.bar(x + i*width, vals, width, label=method, color=palette[i % len(palette)], alpha=0.85)
ax.set_xticks(x + width*(len(all_methods)-1)/2)
ax.set_xticklabels(srcs_list, fontsize=11)
ax.set_ylabel('Mean KS Statistic ↓', fontsize=11)
ax.set_title('Statistical Fidelity (KS) per Source Dataset', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/ks_bar_chart.pdf', bbox_inches='tight')
plt.show(); print('Saved ks_bar_chart.pdf')

In [ ]:
# ── Attack coverage chart ──────────────────────────────────────
method_cov = {'NSG-IDS\n(Fused)': cov_covered}
if 'WGAN-GP (Vanilla)' in baseline_results:
    method_cov['WGAN-GP\n(Vanilla)'] = baseline_results['WGAN-GP (Vanilla)']['cov']
if 'SMOTE' in baseline_results:
    method_cov['SMOTE'] = baseline_results['SMOTE']['cov']

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(method_cov.keys(), method_cov.values(),
              color=['#1565C0','#E65100','#2E7D32','#6A1B9A'][:len(method_cov)], alpha=0.85)
ax.axhline(N_CLASSES, color='red', linestyle='--', linewidth=1.5, label=f'Total ({N_CLASSES} classes)')
ax.bar_label(bars, fmt='%d', padding=2, fontsize=11)
ax.set_ylabel('Attack Classes Covered', fontsize=11)
ax.set_ylim(0, N_CLASSES + 3)
ax.set_title('Attack Class Coverage (≥500 synthetic samples)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/coverage_chart.pdf', bbox_inches='tight')
plt.show(); print('Saved coverage_chart.pdf')

In [ ]:
# ── Synthetic distribution per class ──────────────────────────
cnt_series = pd.Series({label_enc.classes_[c]: cnt.get(c,0) for c in range(N_CLASSES)})
cnt_series = cnt_series.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(5, N_CLASSES*0.35)))
colors_bar = ['#C62828' if v < CFG['COVERAGE_MIN'] else '#1565C0' for v in cnt_series]
cnt_series.plot(kind='barh', ax=ax, color=colors_bar, alpha=0.85)
ax.axvline(CFG['COVERAGE_MIN'], color='red', linestyle='--', linewidth=1.5,
           label=f'Threshold ({CFG["COVERAGE_MIN"]})')
ax.set_xlabel('Synthetic Samples Generated', fontsize=11)
ax.set_title('Synthetic Samples per Attack Class (NSG-IDS Fused)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/class_distribution.pdf', bbox_inches='tight')
plt.show(); print('Saved class_distribution.pdf')

In [ ]:
# ── Feature KS heatmap (per class, per feature) ───────────────
N_FEA_SHOW = min(10, len(CONT_COLS))
N_CLS_SHOW = min(10, N_CLASSES)
ks_matrix  = np.zeros((N_CLS_SHOW, N_FEA_SHOW))

with torch.no_grad():
    for ci in range(N_CLS_SHOW):
        real_ci = X_test[y_test == ci][:500]
        if len(real_ci) < 20: continue
        z  = torch.randn(len(real_ci), CFG['Z_DIM'], device=DEVICE)
        c_ = torch.full((len(real_ci),), ci, dtype=torch.long, device=DEVICE)
        s_ = torch.full((len(real_ci),), class_to_src.get(ci,0), dtype=torch.long, device=DEVICE)
        fake_ci = G_fused(z, c_, s_, 0.2).cpu().numpy()
        for fi, feat in enumerate(CONT_COLS[:N_FEA_SHOW]):
            real_f = decode_to_raw(real_ci)
            fake_f = decode_to_raw(fake_ci)
            if feat in real_f.columns and feat in fake_f.columns:
                r_f, f_f = real_f[feat].dropna(), fake_f[feat].dropna()
                if len(r_f) > 0 and len(f_f) > 0:
                    ks, _ = ks_2samp(r_f, f_f)
                    ks_matrix[ci, fi] = ks

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(ks_matrix,
            xticklabels=[c.replace('_',' ') for c in CONT_COLS[:N_FEA_SHOW]],
            yticklabels=[label_enc.classes_[i] for i in range(N_CLS_SHOW)],
            cmap='YlOrRd', vmin=0, vmax=0.5, ax=ax,
            annot=True, fmt='.2f', linewidths=0.3)
ax.set_title('Per-Class KS Statistic (lower = better fidelity)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('CIF-32 Continuous Feature', fontsize=11)
ax.set_ylabel('Attack Class', fontsize=11)
plt.tight_layout()
plt.savefig('./figures/ks_heatmap.pdf', bbox_inches='tight')
plt.show(); print('Saved ks_heatmap.pdf')

## 12 · Final Results Tables (Paper-Ready)

In [ ]:
print('=' * 65)
print('  FINAL RESULTS — copy these into NSG-IDS.tex (replace \\nv{})')
print('=' * 65)

# Table 3
print('\n--- Table 3: Statistical Fidelity ---')
print(f'{"Method":<22} {"Source":<15} {"KS":>8} {"CD":>8} {"JSD":>8}')
for src, m in sorted(fidelity.items()):
    sn = src_names.get(src, str(src))
    print(f'{"NSG-IDS (Fused)":<22} {sn:<15} {m["KS"]:8.4f} {m["CD"]:8.4f} {m["JSD"]:8.4f}')

# Table 4
print('\n--- Table 4: TSTR Macro-F1 ---')
print(f'{"Method":<22} {"Source":<15} {"F1":>7} {"Prec":>7} {"Rec":>7} {"Acc":>7}')
for src, m in sorted(mle_fused.items()):
    sn = src_names.get(src, str(src))
    print(f'{"NSG-IDS (Fused)":<22} {sn:<15} {m["F1"]:7.4f} {m["Prec"]:7.4f} {m["Rec"]:7.4f} {m["Acc"]:7.4f}')
for src, m in sorted(mle_real.items()):
    sn = src_names.get(src, str(src))
    print(f'{"Real-Data Baseline":<22} {sn:<15} {m["F1"]:7.4f} {m["Prec"]:7.4f} {m["Rec"]:7.4f} {m["Acc"]:7.4f}')

# Table 5
print(f'\n--- Table 5: Coverage ---')
print(f'NSG-IDS (Fused): {cov_covered} / {N_CLASSES}  ({cov_pct:.1f}%)')
if 'WGAN-GP (Vanilla)' in baseline_results:
    v = baseline_results['WGAN-GP (Vanilla)']['cov']
    print(f'WGAN-GP Vanilla: {v} / {N_CLASSES}  ({100*v/N_CLASSES:.1f}%)')

# Table 6
print('\n--- Table 6: Ablation ---')
print(f'{"Configuration":<25} {"F1":>7} {"KS":>7} {"CD":>7} {"Coverage":>8}')
for name, m in ablation_results.items():
    print(f'{name:<25} {m["F1"]:7.4f} {m["KS"]:7.4f} {m["CD"]:7.4f} {m["Coverage"]:8d}')

# Model size
n_params = sum(p.numel() for p in G_fused.parameters())
print(f'\n--- Implementation Details ---')
print(f'Generator parameters: {n_params/1e6:.2f}M')
print(f'Synthetic samples generated: {len(X_syn):,} ({N_CLASSES} classes × {CFG["N_SYNTH_PER_CLASS"]:,})')


# ── Validate results against paper-reported values (Table 3 & 4) ─────────────
# Expected values from the submitted paper. A warning fires if actual results
# deviate by more than the tolerance, signalling a bug in preprocessing or training.
PAPER_EXPECTED = {
    'fidelity': {
        0: {'KS': 0.061, 'CD': 0.175, 'JSD': 0.081},  # CIC-IDS-2017
        1: {'KS': 0.062, 'CD': 0.168, 'JSD': 0.084},  # UNSW-NB15
        2: {'KS': 0.053, 'CD': 0.171, 'JSD': 0.072},  # NF-ToN-IoT
    },
    'tstr_f1': {
        0: 0.869,   # CIC-IDS-2017
        1: 0.851,   # UNSW-NB15
        2: 0.899,   # NF-ToN-IoT
    },
}
KS_TOL, F1_TOL = 0.05, 0.05
print('\n--- Results Validation vs Paper ---')
_any_warn = False
for src, exp in PAPER_EXPECTED['fidelity'].items():
    if src not in fidelity: continue
    act = fidelity[src]
    for metric, exp_val in exp.items():
        diff = abs(act[metric] - exp_val)
        if diff > KS_TOL:
            print(f'  WARNING: {src_names.get(src,src)} {metric}: '
                  f'expected {exp_val:.4f}, got {act[metric]:.4f} (diff={diff:.4f} > tol={KS_TOL})')
            _any_warn = True
for src, exp_f1 in PAPER_EXPECTED['tstr_f1'].items():
    if src not in mle_fused: continue
    act_f1 = mle_fused[src]['F1']
    diff = abs(act_f1 - exp_f1)
    if diff > F1_TOL:
        print(f'  WARNING: {src_names.get(src,src)} TSTR F1: '
              f'expected {exp_f1:.4f}, got {act_f1:.4f} (diff={diff:.4f} > tol={F1_TOL})')
        _any_warn = True
if not _any_warn:
    print('  All metrics within tolerance of paper-reported values.')

# Save results JSON
results_out = dict(
    fidelity={
        src_names.get(s,str(s)): {k: round(v,5) for k,v in m.items()}
        for s,m in fidelity.items()
    },
    tstr={
        src_names.get(s,str(s)): {k: round(v,5) for k,v in m.items()}
        for s,m in mle_fused.items()
    },
    coverage=dict(covered=cov_covered, total=N_CLASSES, pct=round(cov_pct,2)),
    ablation={name: {k: round(v,5) if isinstance(v,float) else v for k,v in m.items()}
              for name,m in ablation_results.items()},
    model_params=n_params,
)
with open('./outputs/results.json', 'w') as f:
    json.dump(results_out, f, indent=2)
print('\nAll results saved to ./outputs/results.json')
print('All figures saved to ./figures/')